# 00 — Data audit (FD001–FD004)

Analysis only: every number below is computed from `data/raw` by this notebook; nothing is
taken from prior knowledge of C-MAPSS and nothing in `src/` or `config.py` is changed.

**Test-set access (CLAUDE.md rule 3).** `test_FD00x.txt` is read only for its truncation
lengths (section A); `RUL_FD00x.txt` only for its label distribution (sections B and C).
No model is fitted on, and no feature is computed from, test data. Every other section uses
the training files only.

**Uncertainty (rule 4).** Statistics of a distribution carry a 95% percentile-bootstrap CI
that resamples *engines* (never rows), with `n` = number of engines. Plain counts over the
fixed files (number of units, engines in a bucket) are exact census values and carry no CI.

**Reproducibility (D34).** Everything runs inside `repro.cpu_deterministic(n_threads=1)`;
the report ends with `repro.render_markdown` and a `reports/data_audit.env.json` sidecar.
Stochastic steps and their seeds are declared in the parameter cell. The piecewise-linear
(hinge) fits in F/G are deterministic: exhaustive grid over the change-point plus closed-form
least squares — no random initialisation, so no seed.

In [ ]:
from __future__ import annotations

import hashlib
import json
from contextlib import ExitStack
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats as sps
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

from turbofan import repro
from turbofan.config import (
    DATASETS,
    KEEP,
    MAINTENANCE_BUCKETS,
    MULTI_REGIME,
    REGIME_KMEANS_N_INIT,
    REGIME_KMEANS_SEED,
    REGIME_N_CLUSTERS,
    RUL_CAP,
    SEQ_LEN,
    SPLIT_FRAC,
    WINDOW,
)
from turbofan.data.loader import load_dataset
from turbofan.evaluation.comparison import split_engines

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
REPORT_PATH = ROOT / "reports" / "data_audit.md"
FIG_DIR = ROOT / "reports" / "figures" / "data_audit"
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Parameters

Analysis parameters of this audit (not pipeline parameters). Each is stated in the report.

In [ ]:
N_THREADS = 1
N_BOOT = 2000  # bootstrap resamples (engines)
CI_LEVEL = 0.95
BOOT_SEED = 20261002  # numpy default_rng, keyed by [BOOT_SEED, section, dataset]
AUDIT_KMEANS_SEED = 0  # D: this audit's own regime clustering
AUDIT_KMEANS_N_INIT = 10
K_RANGE = list(range(2, 11))  # silhouette scan
SIL_SAMPLE = 10_000  # rows sampled for silhouette (O(n^2) otherwise)
SIL_SEED = 0
STABILITY_SEEDS = list(range(10))  # D05: KMeans restarts with config's k / n_init
VAL_STEP = 5  # nb02 make_val_instances default
MIN_SEG_GRID = (5, 10, 20)  # F: min points each side of the hinge (sensitivity)
MIN_SEG_PRIMARY = 10
TAIL_LEN = 30  # H: early/late segment length for the noise-growth ratio
SLOPE_WINDOWS = (15, WINDOW, SEQ_LEN)  # G/H: windows whose slope resolution is reported
SMOOTH_WINDOWS = (5, 15, 30)  # H: rolling-mean windows of the removed 258-feature family
EWM_SPAN = 15  # H: "ewm15" — span assumed from the name; definition not recoverable

SECTIONS = "ABCDEFGHIJKL"  # keys the bootstrap streams; append only
ALPHA = 1 - CI_LEVEL
SEEDS = {
    "bootstrap_default_rng": BOOT_SEED,
    "audit_kmeans_random_state": AUDIT_KMEANS_SEED,
    "silhouette_sample_random_state": SIL_SEED,
    **{f"d05_stability_kmeans_random_state_{s}": s for s in STABILITY_SEEDS},
}

stack = ExitStack()
stack.enter_context(repro.cpu_deterministic(n_threads=N_THREADS))

REPORT: list[str] = []  # markdown chunks, assembled into reports/data_audit.md at the end
FIGS: dict[str, str] = {}  # figure file -> sha256


def rng_for(section: str, ds: str) -> np.random.Generator:
    return np.random.default_rng([BOOT_SEED, SECTIONS.index(section), DATASETS.index(ds)])


def boot(values, rng, stat=np.median):
    """(estimate, lo, hi, n) — percentile bootstrap over the given per-engine values."""
    v = np.asarray(values, dtype=float)
    if np.isnan(v).any():
        raise ValueError("NaN in bootstrap input — drop and report them explicitly first")
    n = len(v)
    idx = rng.integers(0, n, size=(N_BOOT, n))
    bs = stat(v[idx], axis=1)
    lo, hi = np.quantile(bs, [ALPHA / 2, 1 - ALPHA / 2])
    return float(stat(v)), float(lo), float(hi), n


def fmt(b, p=2):
    est, lo, hi, n = b
    return f"{est:.{p}f} [{lo:.{p}f}, {hi:.{p}f}] (n={n})"


NL = "\n"  # newline, for report strings assembled with +


def md_table(df: pd.DataFrame) -> str:
    cols = [str(c).replace("|", r"\|") for c in df.columns]
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    out += [
        "| " + " | ".join(str(v).replace("|", r"\|") for v in row) + " |"
        for row in df.itertuples(index=False)
    ]
    return "\n".join(out)


def save_fig(fig, name: str) -> str:
    path = FIG_DIR / name
    fig.savefig(path, dpi=110, bbox_inches="tight", metadata={"Software": None})
    FIGS[name] = hashlib.sha256(path.read_bytes()).hexdigest()
    return f"figures/data_audit/{name}"


def per_engine_corr(x: pd.DataFrame, y: pd.Series, unit: pd.Series) -> pd.DataFrame:
    """Pearson r per engine between each column of x and y (NaN where a side is constant)."""
    xy = x.assign(_y=y.values)
    g = xy.groupby(unit.values)
    c = xy - g.transform("mean")
    num = c[x.columns].mul(c["_y"], axis=0).groupby(unit.values).sum()
    sxx = (c[x.columns] ** 2).groupby(unit.values).sum()
    syy = (c["_y"] ** 2).groupby(unit.values).sum()
    den = np.sqrt(sxx.mul(syy, axis=0))
    with np.errstate(invalid="ignore", divide="ignore"):
        r = num / den
    return r.where(den > 0)


def per_engine_spearman(x: pd.DataFrame, y: pd.Series, unit: pd.Series) -> pd.DataFrame:
    xr = x.groupby(unit.values).rank(method="average")
    yr = y.groupby(unit.values).rank(method="average")
    return per_engine_corr(xr, yr, unit)

## Load

`ttf` = uncapped time to failure (last train cycle − cycle). The loader's `rul` column is
capped at `RUL_CAP`; it is used only where the pipeline itself uses it (C).

In [ ]:
train, test, test_rul = {}, {}, {}
for ds in DATASETS:
    tr, te, rul = load_dataset(ds, RAW)
    for name, df in (("train", tr), ("test", te)):
        g = df.groupby("unit")["cycle"]
        if not ((g.min() == 1).all() and (g.max() == g.count()).all()):
            raise ValueError(f"{ds} {name}: cycles are not 1..T consecutive per unit")
    tr = tr.sort_values(["unit", "cycle"]).reset_index(drop=True)
    tr["ttf"] = tr.groupby("unit")["cycle"].transform("max") - tr["cycle"]
    train[ds], test[ds], test_rul[ds] = tr, te, rul

SENSORS = [f"s{i}" for i in range(1, 22)]
print({ds: train[ds].shape for ds in DATASETS})

## A. Structure

In [ ]:
rows = []
for ds in DATASETS:
    rng = rng_for("A", ds)
    life = train[ds].groupby("unit")["cycle"].max()
    tlen = test[ds].groupby("unit")["cycle"].max()
    if len(test_rul[ds]) != len(tlen):
        raise ValueError(f"{ds}: {len(test_rul[ds])} RUL labels for {len(tlen)} test units")
    rows.append(
        {
            "dataset": ds,
            "train units": len(life),
            "train cycles/unit min": int(life.min()),
            "train cycles/unit median [95% CI]": fmt(boot(life, rng), 1),
            "train cycles/unit IQR": f"{life.quantile(0.25):.0f}–{life.quantile(0.75):.0f}",
            "train cycles/unit max": int(life.max()),
            "test units": len(tlen),
            "test length min": int(tlen.min()),
            "test length median [95% CI]": fmt(boot(tlen, rng), 1),
            "test length max": int(tlen.max()),
            f"test units < SEQ_LEN ({SEQ_LEN})": int((tlen < SEQ_LEN).sum()),
            f"test units < WINDOW ({WINDOW})": int((tlen < WINDOW).sum()),
        }
    )
A = pd.DataFrame(rows)
A_life = {ds: train[ds].groupby("unit")["cycle"].max() for ds in DATASETS}
A_tlen = {ds: test[ds].groupby("unit")["cycle"].max() for ds in DATASETS}
display(A)

fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharey="row")
for j, ds in enumerate(DATASETS):
    axes[0, j].hist(A_life[ds], bins=30, color="tab:blue")
    axes[0, j].set_title(f"{ds} train lifetime (cycles)")
    axes[1, j].hist(A_tlen[ds], bins=30, color="tab:orange")
    axes[1, j].axvline(SEQ_LEN, color="k", ls="--", label=f"SEQ_LEN={SEQ_LEN}")
    axes[1, j].axvline(WINDOW, color="r", ls=":", label=f"WINDOW={WINDOW}")
    axes[1, j].set_title(f"{ds} test truncation length")
axes[1, 0].legend()
fig_a = save_fig(fig, "A_lengths.png")
plt.show()

n_short = {ds: int((A_tlen[ds] < SEQ_LEN).sum()) for ds in DATASETS}
REPORT.append(
    "## A. Structure\n\n"
    "Counts are exact (census of the files). Medians carry a 95% bootstrap CI over engines.\n\n"
    + md_table(A)
    + f"\n\n![lengths]({fig_a})\n\n"
    + "Test engines shorter than `SEQ_LEN` are left-padded with zeros by the LSTM (D11): "
    + ", ".join(f"{ds} {n_short[ds]}/{len(A_tlen[ds])}" for ds in DATASETS)
    + ". Test engines shorter than `WINDOW` get rolling features from fewer than `WINDOW` "
    "cycles: " + ", ".join(f"{ds} {int((A_tlen[ds] < WINDOW).sum())}" for ds in DATASETS) + ".\n"
)

## B. Test label distribution (labels only — no model)

In [ ]:
rows, B_crit = [], {}
for ds in DATASETS:
    rng = rng_for("B", ds)
    y = test_rul[ds].to_numpy()
    row = {
        "dataset": ds,
        "n": len(y),
        "min": int(y.min()),
        "median [95% CI]": fmt(boot(y, rng), 1),
        "IQR": f"{np.quantile(y, 0.25):.0f}–{np.quantile(y, 0.75):.0f}",
        "max": int(y.max()),
    }
    for name, lo, hi in MAINTENANCE_BUCKETS:
        hi_s = "∞" if np.isinf(hi) else f"{hi:.0f}"
        row[f"{name} [{lo:.0f},{hi_s})"] = int(((y >= lo) & (y < hi)).sum())
    row[f"true RUL > {RUL_CAP:.0f}"] = int((y > RUL_CAP).sum())
    B_crit[ds] = row[next(k for k in row if k.startswith("critical"))]
    rows.append(row)
B = pd.DataFrame(rows)
display(B)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, ds in zip(axes, DATASETS, strict=True):
    ax.hist(test_rul[ds], bins=30, color="tab:purple")
    for _, lo, _ in MAINTENANCE_BUCKETS[1:]:
        ax.axvline(lo, color="k", lw=0.8, ls=":")
    ax.axvline(RUL_CAP, color="r", lw=1.2, label=f"RUL_CAP={RUL_CAP:.0f}")
    ax.set_title(f"{ds} test true RUL")
axes[0].legend()
fig_b = save_fig(fig, "B_test_rul.png")
plt.show()

B_crit_total = sum(B_crit.values())
B_n_total = int(B["n"].sum())
REPORT.append(
    "## B. Test label distribution\n\n"
    "Label distribution only (allowed by rule 3); no prediction is involved.\n\n"
    + md_table(B)
    + f"\n\n![test RUL]({fig_b})\n\n"
    + "**The headline metric (critical-zone RMSE, true RUL in [0, 25)) rests on "
    + ", ".join(f"{B_crit[ds]} engines ({ds})" for ds in DATASETS)
    + f" — {B_crit_total} of {B_n_total} test engines pooled.** Each per-dataset headline "
    "number is an RMSE over that many residuals, one per engine.\n"
)

## C. Validation realism: `make_val_instances` vs test true RUL

`make_val_instances` is copied from `notebooks/02_modeling.ipynb` cell 2 — logic unchanged,
reformatted (it is not in `src/`). It only reads `unit`, `cycle` and the capped `rul`, so
applying it to the raw validation frame gives the same rows as applying it to the feature
frame (add_features neither drops nor reorders rows within an engine). Validation engines are
`split_engines(train, SPLIT_FRAC)` — the pipeline's split.

In [ ]:
def make_val_instances(feat_df, step=5):
    # Sample a truncation instance at every step-th cycle from each val engine's
    # degradation phase (RUL < 125). Covers the full RUL range, matching the
    # test-set distribution. Skips the healthy plateau where RUL is capped.
    parts = []
    for unit, grp in feat_df.groupby("unit"):  # noqa: B007 — kept as in nb02
        degrade = grp[grp["rul"] < RUL_CAP].sort_values("cycle")
        if len(degrade) == 0:
            degrade = grp.sort_values("cycle")
        parts.append(degrade.iloc[::step])
    return pd.concat(parts, ignore_index=True)


def ks_boot(groups_a: list[np.ndarray], b: np.ndarray, rng) -> tuple[float, float, float]:
    """KS statistic between pooled groups_a and b; CI resamples engines on both sides."""
    est = sps.ks_2samp(np.concatenate(groups_a), b).statistic
    na, nb = len(groups_a), len(b)
    bs = np.empty(N_BOOT)
    for i in range(N_BOOT):
        ia = rng.integers(0, na, na)
        ib = rng.integers(0, nb, nb)
        bs[i] = sps.ks_2samp(np.concatenate([groups_a[k] for k in ia]), b[ib]).statistic
    lo, hi = np.quantile(bs, [ALPHA / 2, 1 - ALPHA / 2])
    return float(est), float(lo), float(hi)


rows, C_inst = [], {}
for ds in DATASETS:
    rng = rng_for("C", ds)
    _, va_e = split_engines(train[ds], SPLIT_FRAC)
    va = train[ds][train[ds]["unit"].isin(va_e)]
    n_fallback = int((va.groupby("unit")["rul"].min() >= RUL_CAP).sum())
    inst = make_val_instances(va, step=VAL_STEP)
    C_inst[ds] = inst
    groups = [g["rul"].to_numpy(float) for _, g in inst.groupby("unit")]
    y_te = test_rul[ds].to_numpy(float)
    y_te_lt = y_te[y_te < RUL_CAP]
    ks_all = ks_boot(groups, y_te, rng)
    ks_lt = ks_boot(groups, y_te_lt, rng)
    rows.append(
        {
            "dataset": ds,
            "val engines": len(groups),
            "val instances": len(inst),
            "engines using fallback (no RUL<cap)": n_fallback,
            "val RUL range": f"{inst['rul'].min():.0f}–{inst['rul'].max():.0f}",
            "test engines": len(y_te),
            f"test RUL ≥ {RUL_CAP:.0f} (unreachable by val)": int((y_te >= RUL_CAP).sum()),
            "KS vs all test [95% CI]": f"{ks_all[0]:.3f} [{ks_all[1]:.3f}, {ks_all[2]:.3f}]",
            f"KS vs test RUL<{RUL_CAP:.0f} [95% CI]": (
                f"{ks_lt[0]:.3f} [{ks_lt[1]:.3f}, {ks_lt[2]:.3f}]"
            ),
            f"test n (RUL<{RUL_CAP:.0f})": len(y_te_lt),
            "val median": f"{np.median(inst['rul']):.0f}",
            "test median": f"{np.median(y_te):.0f}",
        }
    )
C = pd.DataFrame(rows)
display(C)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, ds in zip(axes, DATASETS, strict=True):
    for vals, lab in ((C_inst[ds]["rul"], "val instances"), (test_rul[ds], "test true RUL")):
        v = np.sort(np.asarray(vals, float))
        ax.step(v, np.arange(1, len(v) + 1) / len(v), where="post", label=lab)
    ax.axvline(RUL_CAP, color="r", lw=0.8, ls=":")
    ax.set_title(f"{ds} RUL ECDF")
axes[0].legend()
fig_c = save_fig(fig, "C_val_vs_test_ecdf.png")
plt.show()

C_idx = C.set_index("dataset")
n_ge = {ds: int((test_rul[ds] >= RUL_CAP).sum()) for ds in DATASETS}
REPORT.append(
    "## C. Validation realism\n\n"
    f"`make_val_instances` (nb02 cell 2, logic copied unchanged; step = {VAL_STEP}) applied to the "
    f"pipeline's validation engines (`split_engines`, `SPLIT_FRAC` = {SPLIT_FRAC}), compared "
    "with the test true-RUL labels. KS = two-sample Kolmogorov–Smirnov statistic (0 = identical "
    "distributions). The CI resamples validation engines (with all their instances) and test "
    "engines independently; a percentile bootstrap of KS is biased upward, so a point estimate "
    "can sit at or below the CI's lower end.\n\n" + md_table(C) + f"\n\n![ECDF]({fig_c})\n\n"
    "- By construction validation instances have RUL "
    f"{C_inst[DATASETS[0]]['rul'].min():.0f}–{C_inst[DATASETS[0]]['rul'].max():.0f} "
    f"(every {VAL_STEP}th cycle with capped RUL < {RUL_CAP:.0f}), and every validation engine "
    "contributes the same near-uniform spread, so the pooled validation median is "
    + ", ".join(f"{C_idx.loc[ds, 'val median']}" for ds in DATASETS)
    + " vs test medians "
    + ", ".join(f"{C_idx.loc[ds, 'test median']}" for ds in DATASETS)
    + " (FD001–FD004).\n"
    "- Test engines with true RUL ≥ the cap have no validation counterpart: "
    + ", ".join(f"{ds} {n_ge[ds]}/{len(test_rul[ds])}" for ds in DATASETS)
    + ".\n"
    "- Against the full test distribution the KS CI excludes 0 on every dataset; restricted to "
    "test engines with RUL below the cap the distance is smaller (table). nb02's comment that "
    'the instances match "the test-set distribution" holds at best for the RUL < cap part.\n'
)

## D. Operating regimes

KMeans on standardized (op1, op2, op3), train rows only, k = 2..10, silhouette on a seeded
row sample. Inspection: cluster sizes, centers in raw units, and how much a 2-way split
separates the sensors (if a dataset has one regime, a forced split should not move sensor
means). D05: stability of the pipeline's own KMeans settings across random_state.

In [ ]:
D_sil, D_fit, D_scaler, D_sep = {}, {}, {}, {}
for ds in DATASETS:
    X_raw = train[ds][["op1", "op2", "op3"]]
    sc = StandardScaler().fit(X_raw)
    X = sc.transform(X_raw)
    D_scaler[ds] = sc
    sil, fits = {}, {}
    for k in K_RANGE:
        km = KMeans(n_clusters=k, n_init=AUDIT_KMEANS_N_INIT, random_state=AUDIT_KMEANS_SEED)
        lab = km.fit_predict(X)
        fits[k] = km
        sil[k] = float(silhouette_score(X, lab, sample_size=SIL_SAMPLE, random_state=SIL_SEED))
    D_sil[ds], D_fit[ds] = sil, fits
    # forced 2-way split: max over sensors of |mean diff| / pooled within-cluster std
    lab2 = fits[2].labels_
    sens = train[ds][SENSORS]
    eff = {}
    for s in SENSORS:
        a, b = sens[s][lab2 == 0], sens[s][lab2 == 1]
        if a.nunique() == 1 and b.nunique() == 1:
            # single-valued in both clusters: compare values exactly (float means of identical
            # values can differ in the last bit, which would fake an effect)
            eff[s] = 0.0 if a.iloc[0] == b.iloc[0] else np.inf
            continue
        pooled = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
        eff[s] = abs(a.mean() - b.mean()) / pooled
    D_sep[ds] = pd.Series(eff)

sil_df = pd.DataFrame(D_sil).T
sil_df.columns = [f"k={k}" for k in K_RANGE]
display(sil_df.round(4))

raw_spread = pd.DataFrame(
    {ds: train[ds][["op1", "op2", "op3"]].std() for ds in DATASETS}
).T.add_prefix("raw std ")
display(raw_spread)
display(pd.DataFrame({ds: D_sep[ds].sort_values(ascending=False).head(3) for ds in DATASETS}))

fig, ax = plt.subplots(figsize=(7, 3.5))
for ds in DATASETS:
    ax.plot(K_RANGE, [D_sil[ds][k] for k in K_RANGE], marker="o", label=ds)
ax.set_xlabel("k")
ax.set_ylabel("silhouette")
ax.legend()
fig_d = save_fig(fig, "D_silhouette.png")
plt.show()

In [ ]:
D_kstar = {ds: max(D_sil[ds], key=D_sil[ds].get) for ds in DATASETS}
for ds in DATASETS:
    k = D_kstar[ds]
    km = D_fit[ds][k]
    centers = pd.DataFrame(
        D_scaler[ds].inverse_transform(km.cluster_centers_), columns=["op1", "op2", "op3"]
    )
    centers["rows"] = np.bincount(km.labels_, minlength=k)
    print(ds, "argmax-silhouette k =", k)
    display(centers.round(4))

# D05: the pipeline's own KMeans settings, varying only random_state
D05 = {}
for ds in DATASETS:
    X = D_scaler[ds].transform(train[ds][["op1", "op2", "op3"]])
    ref = KMeans(
        REGIME_N_CLUSTERS, n_init=REGIME_KMEANS_N_INIT, random_state=REGIME_KMEANS_SEED
    ).fit_predict(X)
    aris = [
        adjusted_rand_score(
            ref,
            KMeans(REGIME_N_CLUSTERS, n_init=REGIME_KMEANS_N_INIT, random_state=s).fit_predict(X),
        )
        for s in STABILITY_SEEDS
    ]
    D05[ds] = (min(aris), max(aris))
print("D05 ARI vs random_state=0 (min, max):", D05)

### D06: do the normalization fallbacks fire on the pipeline's own train/val split?

Pipeline regime model (k = `REGIME_N_CLUSTERS`, config seed) fit on the train split of each
multi-regime set, applied to its validation split — the exact path `add_features` takes.

In [ ]:
D06 = {}
for ds in sorted(MULTI_REGIME):
    tr_e, va_e = split_engines(train[ds], SPLIT_FRAC)
    trs = train[ds][train[ds]["unit"].isin(tr_e)]
    vas = train[ds][train[ds]["unit"].isin(va_e)]
    sc = StandardScaler().fit(trs[["op1", "op2", "op3"]])
    km = KMeans(
        REGIME_N_CLUSTERS, n_init=REGIME_KMEANS_N_INIT, random_state=REGIME_KMEANS_SEED
    ).fit(sc.transform(trs[["op1", "op2", "op3"]]))
    r_tr = km.predict(sc.transform(trs[["op1", "op2", "op3"]]))
    r_va = km.predict(sc.transform(vas[["op1", "op2", "op3"]]))
    sd = trs[KEEP].groupby(r_tr).std()
    D06[ds] = {
        "train-split rows per regime (min)": int(np.bincount(r_tr).min()),
        "val regimes unseen in train": int(len(set(r_va) - set(r_tr))),
        "KEEP sensor × regime cells with std = 0": int((sd == 0).sum().sum()),
        "KEEP sensor × regime cells with std NaN": int(sd.isna().sum().sum()),
    }
display(pd.DataFrame(D06))

sep_tab = pd.DataFrame(
    {
        ds: {
            "largest forced-split shift (pooled SD)": (
                f"{D_sep[ds].replace(np.inf, np.nan).max():.3f}"
            ),
            "sensor": D_sep[ds].replace(np.inf, np.nan).idxmax(),
            "sensors single-valued per cluster but differing": int(np.isinf(D_sep[ds]).sum()),
        }
        for ds in DATASETS
    }
).T
display(sep_tab)

In [ ]:
# Regimes used by the rest of the audit — chosen by inspecting the output above (see the
# markdown cell after this one and the report's section D): datasets whose op settings vary
# only at measurement-noise scale, with low silhouette at every k and no sensor separated by a
# forced split, are treated as single-regime; the others take the silhouette argmax.
SINGLE_REGIME = ("FD001", "FD003")
AUDIT_K = {ds: 1 if ds in SINGLE_REGIME else D_kstar[ds] for ds in DATASETS}
REG = {}
for ds in DATASETS:
    if AUDIT_K[ds] == 1:
        REG[ds] = np.zeros(len(train[ds]), dtype=int)
    else:
        REG[ds] = D_fit[ds][AUDIT_K[ds]].labels_
print(AUDIT_K)

centers6 = {}
for ds in DATASETS:
    k = D_kstar[ds]
    km = D_fit[ds][k]
    c = pd.DataFrame(
        D_scaler[ds].inverse_transform(km.cluster_centers_), columns=["op1", "op2", "op3"]
    )
    c["train rows"] = np.bincount(km.labels_, minlength=k)
    centers6[ds] = c.sort_values(["op1", "op3"]).round(3)
sil_md = sil_df.round(4).reset_index(names="dataset")
REPORT.append(
    "## D. Operating regimes\n\n"
    f"KMeans (n_init = {AUDIT_KMEANS_N_INIT}, random_state = {AUDIT_KMEANS_SEED}) on standardized "
    f"op1–op3, train rows only. Silhouette on a {SIL_SAMPLE}-row sample "
    f"(random_state = {SIL_SEED}).\n\n" + md_table(sil_md) + f"\n\n![silhouette]({fig_d})\n\n"
    "Raw spread of the operating settings (train rows):\n\n"
    + md_table(raw_spread.round(6).reset_index(names="dataset"))
    + "\n\nForced 2-way split: the largest shift it induces in any non-constant sensor, in "
    "pooled within-cluster SDs (a real regime boundary moves sensors; noise does not):\n\n"
    + md_table(sep_tab.reset_index(names="dataset"))
    + "\n\n**Choice.** "
    + " ".join(
        f"{ds}: single regime — op settings vary by at most {raw_spread.loc[ds].max():.4f} "
        f"(raw-unit SD), best silhouette {max(D_sil[ds].values()):.3f} (k = {D_kstar[ds]}), a "
        "forced split moves no sensor by more than "
        f"{D_sep[ds].replace(np.inf, np.nan).max():.3f} SD."
        for ds in SINGLE_REGIME
    )
    + " "
    + " ".join(
        f"{ds}: k = {AUDIT_K[ds]} (silhouette {D_sil[ds][AUDIT_K[ds]]:.4f}; next best "
        f"{sorted(D_sil[ds].values())[-2]:.4f})."
        for ds in DATASETS
        if ds not in SINGLE_REGIME
    )
    + "\n\nCluster centers at the silhouette argmax, raw units:\n\n"
    + "\n\n".join(
        f"**{ds}**\n\n" + md_table(centers6[ds]) for ds in DATASETS if ds not in SINGLE_REGIME
    )
    + "\n\n**D05 — stability of the pipeline's own regime model** (k = "
    f"{REGIME_N_CLUSTERS}, n_init = {REGIME_KMEANS_N_INIT}) across random_state "
    f"{STABILITY_SEEDS[0]}–{STABILITY_SEEDS[-1]}: adjusted Rand index vs random_state = "
    f"{REGIME_KMEANS_SEED}, min–max: "
    + ", ".join(f"{ds} {D05[ds][0]:.3f}–{D05[ds][1]:.3f}" for ds in DATASETS)
    + " (FD001/FD003 listed for completeness; the pipeline does not cluster them).\n\n"
    "**D06 — do the normalization fallbacks fire on the pipeline's train/val split?** "
    "(regime model fit on the train split, applied to the validation split, as `add_features` "
    "does):\n\n" + md_table(pd.DataFrame(D06).reset_index(names="check")) + "\n"
)

## E. Sensor informativeness

Within-regime z-score (regime mean/std over all train rows of the dataset; regimes from D).
Per engine, Spearman ρ between the normalized sensor and `ttf`.

- **constant**: one unique value within every regime (no within-regime information).
- **flat**: the median per-engine |ρ| does not exceed a noise baseline — the same statistic
  with `ttf` randomly permuted within each engine (seeded) — i.e. the 95% CIs overlap.
- **informative**: median |ρ| CI lies above the baseline CI. Split by direction: *consistent*
  if at least `SIGN_SHARE` of engines share the majority sign of ρ, else *mixed* (the
  sensor trends up in some engines and down in others). |ρ| is used for the informativeness
  test because a median of signed ρ is ~0 for a strongly but bi-directionally trending sensor.

In [ ]:
SIGN_SHARE = 0.9  # E: share of engines agreeing in direction for "consistent" (audit choice)


def regime_z(
    df: pd.DataFrame, cols: list[str], reg: np.ndarray
) -> tuple[pd.DataFrame, pd.DataFrame]:
    g = df[cols].groupby(reg)
    single = g.transform("nunique") == 1
    z = (df[cols] - g.transform("mean")) / g.transform("std")
    # A sensor single-valued within a regime carries no variation there: z = 0 by definition
    # (reported as "constant" when that holds in every regime).
    return z.mask(single, 0.0), single


Z, E_rho, E_cls, E_tab, E_dir = {}, {}, {}, {}, {}
for ds in DATASETS:
    rng = rng_for("E", ds)
    df = train[ds]
    z, single = regime_z(df, SENSORS, REG[ds])
    Z[ds] = z
    rho = per_engine_spearman(z, df["ttf"], df["unit"])
    ttf_perm = df.groupby("unit")["ttf"].transform(lambda v, rng=rng: rng.permutation(v.to_numpy()))
    rho_null = per_engine_spearman(z, ttf_perm, df["unit"])
    E_rho[ds] = rho
    var_by_reg = df[SENSORS].groupby(REG[ds]).var().mask(single.groupby(REG[ds]).all(), 0.0)
    within_var = var_by_reg.mul(np.bincount(REG[ds]) - 1, axis=0).sum() / (len(df) - AUDIT_K[ds])
    rows = []
    for s in SENSORS:
        const_all = bool(single[s].all())
        r = rho[s].dropna()
        if const_all:
            cls, b_abs, b_null, b_med, share = "constant", None, None, None, None
        else:
            b_abs = boot(r.abs(), rng)
            b_null = boot(rho_null[s].dropna().abs(), rng)
            b_med = boot(r, rng)
            maj = np.sign(b_med[0])
            share = boot((np.sign(r) == maj).astype(float), rng, stat=np.mean)
            if b_abs[1] <= b_null[2]:
                cls = "flat"
            else:
                cls = "informative, consistent" if share[0] >= SIGN_SHARE else "informative, mixed"
            E_dir[ds, s] = -maj  # direction of change toward failure (ρ is vs time-to-failure)
        if AUDIT_K[ds] == 1:
            wt = "n/a (1 regime)"
        elif df[s].nunique() == 1:
            wt = "—"
        else:
            wt = f"{within_var[s] / df[s].var():.3f}"
        rows.append(
            {
                "sensor": s,
                "within-regime std (raw units)": f"{np.sqrt(within_var[s]):.4g}",
                "within/total variance": wt,
                "median ρ(z, ttf) [95% CI]": fmt(b_med, 3) if b_med else "—",
                "ρ IQR across engines": f"{r.quantile(0.25):.3f} to {r.quantile(0.75):.3f}"
                if b_med
                else "—",
                "median |ρ| [95% CI]": fmt(b_abs, 3).rsplit(" (", 1)[0] if b_abs else "—",
                "noise baseline median |ρ| [95% CI]": fmt(b_null, 3).rsplit(" (", 1)[0]
                if b_null
                else "—",
                "share of engines with majority sign": fmt(share, 2).rsplit(" (", 1)[0]
                if share
                else "—",
                "engines with ρ undefined": int(rho[s].isna().sum()),
                "class": cls,
                "in KEEP": "yes" if s in KEEP else "no",
            }
        )
        E_cls[ds, s] = cls
    E_tab[ds] = pd.DataFrame(rows)
    display(Markdown(f"**{ds}** (regimes = {AUDIT_K[ds]}, engines = {df['unit'].nunique()})"))
    display(E_tab[ds])

E_inf = {ds: [s for s in SENSORS if E_cls[ds, s].startswith("informative")] for ds in DATASETS}
E_cons = {ds: [s for s in SENSORS if E_cls[ds, s] == "informative, consistent"] for ds in DATASETS}
E_dis = {
    ds: {
        "informative, not in KEEP": [s for s in E_inf[ds] if s not in KEEP],
        "in KEEP, not informative": [s for s in KEEP if s not in E_inf[ds]],
    }
    for ds in DATASETS
}
display(pd.DataFrame(E_dis).T)

# D03: does per-regime normalization matter? Spearman per engine on raw values (= global
# z-score, an affine map) vs on within-regime z.
E_d03 = {}
for ds in DATASETS:
    rng = rng_for("E", ds)
    df = train[ds]
    rho_raw = per_engine_spearman(df[KEEP], df["ttf"], df["unit"])
    d = (E_rho[ds][KEEP].abs() - rho_raw.abs()).mean(axis=1)
    E_d03[ds] = (
        boot(rho_raw[KEEP].abs().mean(axis=1).dropna(), rng),
        boot(E_rho[ds][KEEP].abs().mean(axis=1).dropna(), rng),
        boot(d.dropna(), rng),
    )
for ds, (a, b_, d) in E_d03.items():
    print(ds, "mean|ρ| raw/global:", fmt(a, 3), " regime-z:", fmt(b_, 3), " Δ:", fmt(d, 3))

fig, ax = plt.subplots(figsize=(10, 3.2))
heat = pd.DataFrame({ds: E_rho[ds].median() for ds in DATASETS}).T
im = ax.imshow(heat.to_numpy(float), cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(SENSORS)), SENSORS)
ax.set_yticks(range(len(DATASETS)), DATASETS)
fig.colorbar(im, label="median per-engine ρ(z, ttf)")
fig_e = save_fig(fig, "E_spearman_heatmap.png")
plt.show()

d03_tab = pd.DataFrame(
    [
        {
            "dataset": ds,
            "mean |ρ| over KEEP, raw (= global z)": fmt(a, 3),
            "mean |ρ| over KEEP, within-regime z": fmt(b_, 3),
            "paired Δ": fmt(d, 3),
        }
        for ds, (a, b_, d) in E_d03.items()
    ]
)
dis_tab = pd.DataFrame(
    [
        {
            "dataset": ds,
            **{k: ", ".join(v) or "—" for k, v in E_dis[ds].items()},
            "KEEP sensors with mixed direction": ", ".join(
                s for s in KEEP if E_cls[ds, s] == "informative, mixed"
            )
            or "—",
            "direction-consistent informative (used in F)": ", ".join(E_cons[ds]),
        }
        for ds in DATASETS
    ]
)
REPORT.append(
    "## E. Sensor informativeness\n\n"
    "Within-regime z-score with the regimes chosen in D. ρ = per-engine Spearman correlation "
    "between the normalized sensor and time to failure (uncapped). Classes: *constant* = a "
    "single value within every regime; *flat* = median |ρ| CI overlaps the noise baseline "
    "(`ttf` permuted within each engine); *informative* otherwise — *consistent* if ≥ "
    f"{SIGN_SHARE:.0%} of engines share the majority sign, else *mixed* (the sensor rises in some "
    "engines and falls in others).\n\n"
    + "\n\n".join(
        f"**{ds}** — {AUDIT_K[ds]} regime(s), {train[ds]['unit'].nunique()} engines\n\n"
        + md_table(E_tab[ds])
        for ds in DATASETS
    )
    + f"\n\n![heatmap]({fig_e})\n\n"
    "**Disagreements with `config.KEEP`:**\n\n"
    + md_table(dis_tab)
    + "\n\n**Per-regime vs global normalization (D03).** Spearman ρ is invariant to a global "
    "affine map, so ρ on raw values equals ρ on a global z-score:\n\n" + md_table(d03_tab) + "\n"
)

## F. RUL-cap evidence: per-engine change-point

Health signal per dataset: mean of the *direction-consistent* informative sensors'
within-regime z (E), each signed so it rises toward failure. Mixed-direction sensors are left
out: one orientation would cancel them in a share of engines. Per engine, fit
`y = a + b·max(0, t − τ)` (flat, then linear) by exhaustive search over τ with at least
`min_seg` cycles on each side; closed-form least squares for (a, b). Change-point in
remaining cycles = `ttf` at τ. Engines whose best fit has b ≤ 0 (no rise) are counted and
excluded; engines whose τ sits at the earliest allowed cycle (no flat phase found) are
counted. Repeated per direction-consistent sensor at `MIN_SEG_PRIMARY` as a robustness
check (engines whose fitted slope points away from failure are excluded and counted).

In [ ]:
def hinge_fit(t: np.ndarray, y: np.ndarray, min_seg: int):
    n = len(t)
    js = np.arange(min_seg - 1, n - min_seg)
    taus = t[js]
    H = np.maximum(0.0, t[None, :] - taus[:, None])
    Hc = H - H.mean(axis=1, keepdims=True)
    yc = y - y.mean()
    sxx = (Hc**2).sum(axis=1)
    sxy = Hc @ yc
    b = sxy / sxx
    rss = (yc**2).sum() - b * sxy
    i = int(np.argmin(rss))
    lin_b = np.dot(t - t.mean(), yc) / np.dot(t - t.mean(), t - t.mean())
    lin_rss = (yc**2).sum() - lin_b * np.dot(t - t.mean(), yc)
    return {
        "tau": float(taus[i]),
        "b": float(b[i]),
        "at_start": bool(i == 0),
        "rss": float(rss[i]),
        "lin_rss": float(lin_rss),
    }


def fit_engines(df: pd.DataFrame, signal: pd.Series, min_seg: int) -> pd.DataFrame:
    out = []
    for u, g in df.groupby("unit"):
        t = g["cycle"].to_numpy(float)
        f = hinge_fit(t, signal.loc[g.index].to_numpy(float), min_seg)
        f["unit"] = u
        f["T"] = t[-1]
        f["onset_ttf"] = t[-1] - f["tau"]
        out.append(f)
    return pd.DataFrame(out).set_index("unit")


HI, F_fits = {}, {}
for ds in DATASETS:
    signs = pd.Series({s: E_dir[ds, s] for s in E_cons[ds]})
    HI[ds] = (Z[ds][E_cons[ds]] * signs).mean(axis=1)
    for m in MIN_SEG_GRID:
        F_fits[ds, m] = fit_engines(train[ds], HI[ds], m)

QS = (0.10, 0.25, 0.50, 0.75, 0.90)
rows, F_onset, F_num = [], {}, {}
for ds in DATASETS:
    for m in MIN_SEG_GRID:
        rng = rng_for("F", ds)
        f = F_fits[ds, m]
        ok = f[f["b"] > 0]
        on = ok["onset_ttf"].to_numpy()
        F_onset[ds, m] = on
        row = {
            "dataset": ds,
            "min_seg": m,
            "engines": len(f),
            "b ≤ 0 (excluded)": int((f["b"] <= 0).sum()),
            "no flat phase (τ at start)": int(ok["at_start"].sum()),
            "median RSS hinge/linear": f"{np.median(ok['rss'] / ok['lin_rss']):.3f}",
        }
        qb = {
            q: boot(on, rng, stat=lambda a, axis=None, q=q: np.quantile(a, q, axis=axis))
            for q in QS
        }
        sb = boot((on <= RUL_CAP).astype(float), rng, stat=np.mean)
        F_num[ds, m] = (qb, sb)
        for q in QS:
            row[f"p{int(q * 100)} [95% CI]"] = f"{qb[q][0]:.0f} [{qb[q][1]:.0f}, {qb[q][2]:.0f}]"
        row[f"share onset ≤ {RUL_CAP:.0f} [95% CI]"] = f"{sb[0]:.2f} [{sb[1]:.2f}, {sb[2]:.2f}]"
        rows.append(row)
F = pd.DataFrame(rows)
display(F)

# per-sensor fits (unoriented z) at MIN_SEG_PRIMARY: reused by G for every informative sensor;
# F's robustness table uses the direction-consistent ones, oriented toward failure.
F_sens, F_sens_fits = {}, {}
for ds in DATASETS:
    rng = rng_for("F", ds)
    rows = []
    for s in E_inf[ds]:
        F_sens_fits[ds, s] = fit_engines(train[ds], Z[ds][s], MIN_SEG_PRIMARY)
    for s in E_cons[ds]:
        fs = F_sens_fits[ds, s]
        # engines where the raw sensor never changes have no trend to date (their fitted
        # slope is float noise around 0): excluded and counted
        varies = (train[ds].groupby("unit")[s].nunique() > 1).reindex(fs.index)
        fs_v = fs[varies.to_numpy()]
        ok = fs_v[np.sign(fs_v["b"]) == E_dir[ds, s]]
        rows.append(
            {
                "sensor": s,
                "engines where sensor varies": f"{len(fs_v)}/{len(fs)}",
                "of which slope toward failure": len(ok),
                "median onset ttf [95% CI]": fmt(boot(ok["onset_ttf"], rng), 0),
            }
        )
    F_sens[ds] = pd.DataFrame(rows)
    display(Markdown(f"**{ds}**"))
    display(F_sens[ds])

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, ds in zip(axes, DATASETS, strict=True):
    ax.hist(F_onset[ds, MIN_SEG_PRIMARY], bins=30, color="tab:green")
    ax.axvline(RUL_CAP, color="r", ls="--", label=f"RUL_CAP={RUL_CAP:.0f}")
    ax.set_title(f"{ds} change-point (remaining cycles)")
axes[0].legend()
fig_f = save_fig(fig, "F_changepoint.png")
plt.show()

Fq = {ds: F_num[ds, MIN_SEG_PRIMARY] for ds in DATASETS}
med_lo = min(Fq[ds][0][0.5][1] for ds in DATASETS)
med_hi = max(Fq[ds][0][0.5][2] for ds in DATASETS)
same_across_minseg = all(
    np.array_equal(F_onset[ds, m], F_onset[ds, MIN_SEG_PRIMARY])
    for ds in DATASETS
    for m in MIN_SEG_GRID
)
REPORT.append(
    "## F. RUL-cap evidence: per-engine change-point\n\n"
    "Health signal = mean of the direction-consistent informative sensors (E), oriented to rise "
    "toward failure. Per engine, `y = a + b·max(0, t − τ)` fitted by exhaustive search over τ "
    "(closed-form least squares; deterministic, no seed). Onset = remaining cycles at τ. "
    "Quantiles are across engines with a 95% bootstrap CI over engines.\n\n"
    + md_table(F)
    + "\n\n"
    + (
        "The minimum-segment constraint never binds: onsets are identical for min_seg "
        f"{', '.join(map(str, MIN_SEG_GRID))}.\n\n"
        if same_across_minseg
        else "Onsets differ across min_seg (see table).\n\n"
    )
    + f"![change-point]({fig_f})\n\n"
    "Per-sensor check (each direction-consistent sensor alone, min_seg = "
    f"{MIN_SEG_PRIMARY}):\n\n"
    + "\n\n".join(f"**{ds}**\n\n" + md_table(F_sens[ds]) for ds in DATASETS)
    + "\n\n**Defensible range (not a choice).** Median onset per dataset: "
    + ", ".join(
        f"{ds} {Fq[ds][0][0.5][0]:.0f} [{Fq[ds][0][0.5][1]:.0f}, {Fq[ds][0][0.5][2]:.0f}]"
        for ds in DATASETS
    )
    + f" — a cap at the typical engine's onset lies in **{med_lo:.0f}–{med_hi:.0f}** cycles. "
    "Middle half of engines (p25–p75 point estimates): "
    + ", ".join(f"{ds} {Fq[ds][0][0.25][0]:.0f}–{Fq[ds][0][0.75][0]:.0f}" for ds in DATASETS)
    + "; p90: "
    + ", ".join(f"{ds} {Fq[ds][0][0.9][0]:.0f}" for ds in DATASETS)
    + ". A cap meant to cover most engines' onsets sits at p75–p90, "
    f"**{min(Fq[ds][0][0.75][0] for ds in DATASETS):.0f}–"
    f"{max(Fq[ds][0][0.9][0] for ds in DATASETS):.0f}**. "
    f"The current cap {RUL_CAP:.0f} is at or above the onset of "
    + ", ".join(f"{ds} {Fq[ds][1][0]:.0%}" for ds in DATASETS)
    + " of engines.\n\n"
    "Caveats: a hinge fit dates the onset once the drift is resolvable above noise, so it "
    "estimates *detectable* onset, not physical onset; high-SNR sensors agree with the health "
    "signal, while low-SNR sensors (G) give very different onsets (per-sensor tables).\n"
)

## G. Noise vs trend (informative sensors, within-regime z units)

Per engine: noise σ = std(first difference)/√2 (white-noise estimator, insensitive to slow
trends); trend from the per-sensor hinge fit in F (unoriented, so mixed-direction sensors are
included): slope magnitude |b| per cycle and end-of-life change Δ = |b|·(T − τ).
Reported per sensor as the median across engines with CI: SNR = Δ/σ, cycles
for the trend to accumulate 1σ (σ/b), the lag-1 autocorrelation of the first difference
(−0.5 for white noise around a smooth trend), and the t-ratio of a least-squares slope over
a w-cycle window, b / (σ·√(12 / (w(w²−1)))), for w in `SLOPE_WINDOWS`.

In [ ]:
def diff_stats(z: pd.Series, unit: pd.Series) -> pd.DataFrame:
    out = []
    for u, v in z.groupby(unit.values):
        d = np.diff(v.to_numpy(float))
        ac = np.corrcoef(d[1:], d[:-1])[0, 1] if d.std() > 0 else np.nan
        out.append({"unit": u, "sigma": d.std(ddof=1) / np.sqrt(2), "ac1": ac})
    return pd.DataFrame(out).set_index("unit")


G_rows, G_eng = {}, {}
for ds in DATASETS:
    rng = rng_for("G", ds)
    rows = []
    for s in E_inf[ds]:
        dsx = diff_stats(Z[ds][s], train[ds]["unit"])
        fs = F_sens_fits[ds, s].join(dsx)
        const_eng = fs["sigma"] == 0  # sensor never changes within this engine
        if (fs.loc[~const_eng, "b"] == 0).any():
            raise ValueError(f"{ds} {s}: zero fitted slope in an engine where the sensor varies")
        n_const_eng = int(const_eng.sum())
        fs = fs[~const_eng].copy()
        fs["absb"] = fs["b"].abs()
        fs["delta"] = fs["absb"] * (fs["T"] - fs["tau"])
        fs["snr"] = fs["delta"] / fs["sigma"]
        fs["cyc_1sigma"] = fs["sigma"] / fs["absb"]
        for w in SLOPE_WINDOWS:
            fs[f"t{w}"] = fs["absb"] / (fs["sigma"] * np.sqrt(12 / (w * (w**2 - 1))))
        G_eng[ds, s] = fs
        row = {
            "sensor": s,
            "class": E_cls[ds, s].replace("informative, ", ""),
            "engines excluded (constant within engine)": n_const_eng,
            "σ (z)": fmt(boot(fs["sigma"], rng), 3),
            "Δ EOL change (z)": fmt(boot(fs["delta"], rng), 2),
            "SNR Δ/σ": fmt(boot(fs["snr"], rng), 1),
            "cycles to 1σ": fmt(boot(fs["cyc_1sigma"], rng), 0),
            "lag-1 ac(diff)": fmt(boot(fs["ac1"].dropna(), rng), 2),
        }
        for w in SLOPE_WINDOWS:
            row[f"slope t-ratio w={w}"] = fmt(boot(fs[f"t{w}"], rng), 2).rsplit(" (", 1)[0]
        rows.append(row)
    G_rows[ds] = pd.DataFrame(rows)
    display(Markdown(f"**{ds}**"))
    display(G_rows[ds])

REPORT.append(
    "## G. Noise vs trend\n\n"
    "All informative sensors, within-regime z units, per engine then median across engines "
    "(95% bootstrap CI over engines). σ = std(first difference)/√2; trend from the per-sensor "
    "hinge fit (F, unoriented): |b| per cycle, Δ = |b|·(T − τ). *Cycles to 1σ* = σ/|b|; "
    "*slope t-ratio* = |b| / (σ·√(12 / (w(w² − 1)))), the signal-to-noise of a least-squares "
    f"slope over a w-cycle window (w = {', '.join(map(str, SLOPE_WINDOWS))}: 15 from the removed "
    "family, then `WINDOW` and `SEQ_LEN`). Lag-1 autocorrelation of the first difference is "
    "−0.5 for white noise around a smooth trend; when it holds, averaging over w cycles cuts σ "
    "by √w.\n\n"
    + "\n\n".join(f"**{ds}**\n\n" + md_table(G_rows[ds]) for ds in DATASETS)
    + "\n\nThis section reports inputs to `WINDOW` / `SEQ_LEN`; it does not choose them.\n"
)

## H. The removed 258-feature family

Names recovered from `feature_names_` of `models/xgboost_FD001_base.pkl` (listed in
`reports/artifact_inventory.md` §C); no producing code survives in the working tree or git
history, so definitions below are read from the names. The list is embedded so this notebook
does not depend on the untracked `models/` directory.

In [ ]:
FAMILY_258 = {
    "raw value": "sensor_N — the sensor reading (scaling unknown)",
    "rolling mean w∈{5,15,30}": "sensor_N_rmean_w — trailing mean over w cycles",
    "rolling std w∈{5,15,30}": "sensor_N_rstd_w — trailing std over w cycles",
    "rolling min/max w∈{5,15,30}": "sensor_N_rmin_w / rmax_w — trailing extremes over w cycles",
    "slope w∈{15,30}": "sensor_N_slope_w — trailing linear-trend slope over w cycles",
    "first difference": "sensor_N_diff1 — x_t − x_{t−1}",
    "EWM span 15": "sensor_N_ewm15 — exponentially weighted mean (span assumed from the name)",
    "tst": "sensor_N_tst — definition not recoverable (no code, no docs)",
    "cycle_raw / cycle_norm": "elapsed cycle; cycle_norm = cycle / max_train_cycle (362 in the "
    "FD001 artifact)",
    "interactions": "sensor_11×sensor_4, sensor_11×sensor_12, sensor_4×sensor_12",
    "health_index": "first principal component of the (normalized) kept sensors",
}
assert 14 * 18 + 2 + 3 + 1 == 258  # 18 per-sensor columns × 14 sensors + 6 globals

H_rows = []
for ds in DATASETS:
    rng = rng_for("H", ds)
    df, z = train[ds], Z[ds]
    unit = df["unit"]
    sens = [s for s in KEEP if E_cls[ds, s] != "constant"]
    rho0 = per_engine_spearman(z[sens], df["ttf"], unit).abs()
    if rho0.isna().any().any():
        raise ValueError(f"{ds}: a KEEP sensor is constant within some engine")

    # smoothing: rolling means and EWM vs the raw normalized value
    def smooth_gain(fn, z=z, sens=sens, unit=unit, df=df, rho0=rho0, rng=rng):
        sm = z[sens].groupby(unit.values).transform(fn)
        r = per_engine_spearman(sm, df["ttf"], unit).abs()
        return boot((r - rho0).mean(axis=1).dropna(), rng, stat=np.median)

    for w in SMOOTH_WINDOWS:
        b = smooth_gain(lambda v, w=w: v.rolling(w, min_periods=1).mean())
        H_rows.append(
            (
                ds,
                f"rolling mean w={w}",
                "noise large relative to per-cycle trend → "
                "smoothing raises monotone association with ttf",
                "median per-engine Δ|ρ(·,ttf)| vs raw z, mean over KEEP sensors",
                fmt(b, 3),
            )
        )
    b = smooth_gain(lambda v: v.ewm(span=EWM_SPAN, adjust=True).mean())
    H_rows.append(
        (
            ds,
            f"EWM span {EWM_SPAN}",
            "same as rolling mean",
            "median per-engine Δ|ρ(·,ttf)| vs raw z, mean over KEEP sensors",
            fmt(b, 3),
        )
    )

    # rolling std: does noise grow toward failure?  diff-std last TAIL_LEN vs first TAIL_LEN
    ratios, skews, kurts = [], [], []
    for _, g in z[sens].groupby(unit.values):
        d = g.diff().iloc[1:]
        early, late = d.iloc[:TAIL_LEN], d.iloc[-TAIL_LEN:]
        ratios.append(np.exp(np.log(late.std() / early.std()).mean()))
        skews.append(np.nanmean(sps.skew(d, axis=0)))
        kurts.append(np.nanmean(sps.kurtosis(d, axis=0)))
    H_rows.append(
        (
            ds,
            "rolling std",
            "noise level changes as the engine degrades",
            f"per-engine geo-mean over sensors of std(diff) last {TAIL_LEN} / first "
            f"{TAIL_LEN} cycles (1 = no change)",
            fmt(boot(ratios, rng), 2),
        )
    )
    H_rows.append(
        (
            ds,
            "rolling min/max",
            "non-Gaussian residuals (skew/heavy tails) — else "
            "min/max ≈ mean ± c·std and add nothing",
            "per-engine mean skewness of diff",
            fmt(boot(skews, rng), 3),
        )
    )
    H_rows.append(
        (
            ds,
            "rolling min/max",
            "(as above)",
            "per-engine mean excess kurtosis of diff (0 = Gaussian)",
            fmt(boot(kurts, rng), 3),
        )
    )

    # slope: resolvable within the window?  t-ratio from G, median over informative KEEP sensors
    inf_keep = [s for s in sens if s in E_inf[ds]]
    for w in SLOPE_WINDOWS:
        per_eng = pd.concat([G_eng[ds, s][f"t{w}"] for s in inf_keep], axis=1).median(axis=1)
        H_rows.append(
            (
                ds,
                f"slope w={w}",
                "end-of-life trend is resolvable above noise within w cycles",
                "per-engine median over informative KEEP sensors of slope t-ratio",
                fmt(boot(per_eng.dropna(), rng), 2),
            )
        )

    # diff1: per-cycle change informative?
    dz = z[sens].groupby(unit.values).diff()
    keep_rows = dz.notna().all(axis=1)
    r_d = per_engine_spearman(dz[keep_rows], df["ttf"][keep_rows], unit[keep_rows]).abs()
    H_rows.append(
        (
            ds,
            "first difference",
            "per-cycle change carries information beyond noise",
            "per-engine mean over sensors |ρ(diff1, ttf)|",
            fmt(boot(r_d.mean(axis=1).dropna(), rng), 3),
        )
    )
    ac = pd.concat([G_eng[ds, s]["ac1"] for s in inf_keep], axis=1).mean(axis=1)
    H_rows.append(
        (
            ds,
            "first difference",
            "(as above)",
            "per-engine mean lag-1 autocorrelation of diff (−0.5 = white noise)",
            fmt(boot(ac.dropna(), rng), 3),
        )
    )

    # cycle: elapsed cycles informative across engines only if lifetimes are homogeneous
    life = A_life[ds].to_numpy(float)
    H_rows.append(
        (
            ds,
            "cycle_raw / cycle_norm",
            "lifetimes homogeneous enough that elapsed "
            "cycles predict ttf (cycle_norm is a linear rescale: same ranks)",
            "coefficient of variation of train lifetime",
            fmt(
                boot(
                    life,
                    rng,
                    stat=lambda a, axis=None: a.std(axis=axis, ddof=1) / a.mean(axis=axis),
                ),
                3,
            ),
        )
    )

    # interactions: s4, s11, s12 redundancy and product gain (raw values; definition unknown)
    rr = {}
    for a_, b_ in (("s11", "s4"), ("s11", "s12"), ("s4", "s12")):
        rr[a_, b_] = per_engine_spearman(z[[a_]], z[b_], unit)[a_].dropna()
        H_rows.append(
            (
                ds,
                f"interaction {a_}×{b_}",
                "non-redundant sensors whose product adds "
                "information (if |ρ| between them ≈ 1 the product is a monotone "
                "re-expression of either one)",
                f"median per-engine ρ({a_}, {b_}), within-regime z",
                fmt(boot(rr[a_, b_], rng), 3),
            )
        )

    # health index: shared degradation factor?
    zs = z[sens].to_numpy(float)
    units_arr = unit.to_numpy()
    uniq = np.unique(units_arr)
    S = np.stack([zs[units_arr == u].sum(0) for u in uniq])
    C2 = np.stack([zs[units_arr == u].T @ zs[units_arr == u] for u in uniq])
    N = np.array([(units_arr == u).sum() for u in uniq], float)

    def evr1(ix, N=N, S=S, C2=C2):
        n_, s_, c_ = N[ix].sum(), S[ix].sum(0), C2[ix].sum(0)
        cov = c_ / n_ - np.outer(s_ / n_, s_ / n_)
        ev = np.linalg.eigvalsh(cov)
        return ev[-1] / ev.sum()

    est = evr1(np.arange(len(uniq)))
    bs = np.array([evr1(rng.integers(0, len(uniq), len(uniq))) for _ in range(N_BOOT)])
    lo, hi = np.quantile(bs, [ALPHA / 2, 1 - ALPHA / 2])
    H_rows.append(
        (
            ds,
            "health_index (PCA-1)",
            "kept sensors share one dominant degradation factor",
            "PC1 explained-variance ratio of within-regime z (pooled rows)",
            f"{est:.3f} [{lo:.3f}, {hi:.3f}] (n={len(uniq)})",
        )
    )
    cov = (C2.sum(0) / N.sum()) - np.outer(S.sum(0) / N.sum(), S.sum(0) / N.sum())
    w1 = np.linalg.eigh(cov)[1][:, -1]
    pc1 = pd.DataFrame({"pc1": zs @ w1}, index=df.index)
    r_pc = per_engine_spearman(pc1, df["ttf"], unit)["pc1"].abs()
    best = rho0.median().idxmax()
    gain = (r_pc - rho0[best]).dropna()
    H_rows.append(
        (
            ds,
            "health_index (PCA-1)",
            "(as above)",
            f"median per-engine |ρ(PC1, ttf)| − |ρ(best single sensor {best}, ttf)|",
            fmt(boot(gain, rng), 3),
        )
    )
    H_rows.append((ds, "tst", "unknown", "not measurable: definition not recoverable", "—"))

H = pd.DataFrame(
    H_rows,
    columns=[
        "dataset",
        "family",
        "motivating property",
        "measurement",
        "value [95% CI] (n engines)",
    ],
)
display(H)

fam_tab = pd.DataFrame(
    [{"family": k, "definition (from feature names)": v} for k, v in FAMILY_258.items()]
)
REPORT.append(
    "## H. The removed 258-feature family\n\n"
    "Names recovered from `feature_names_` in `models/xgboost_FD001_base.pkl` (inventoried in "
    "`reports/artifact_inventory.md` §C). No producing code exists in the working tree or git "
    "history, so definitions are read from the names. Per sensor (the same 14 as `KEEP`): raw, "
    "rmean/rstd/rmin/rmax × {5, 15, 30}, slope × {15, 30}, diff1, ewm15, tst = 18 × 14 = 252; "
    "plus cycle_raw, cycle_norm, 3 interactions and health_index = 258.\n\n"
    + md_table(fam_tab)
    + "\n\nFor each family: the signal property that would motivate it, and whether the data "
    "show it (train engines only; within-regime z from D/E; per-engine statistics, median and "
    "95% bootstrap CI over engines). No model is fitted.\n\n"
    + md_table(H)
    + "\n\nCaveat on the multi-regime rows: the late/early noise ratio and the excess kurtosis "
    "of first differences are computed on within-regime z, where consecutive cycles usually fall "
    "in different regimes; if degradation shifts a sensor by a different number of regime SDs "
    "in different regimes, that alone inflates late-life differences. This audit does not "
    "separate that effect from genuine noise growth.\n"
)

## I. Engine subpopulations by degradation direction

Per engine, the sign vector of ρ(sensor, ttf) over the informative sensors of E that are
defined in every engine (sensors constant within some engines are left out and listed).
A sensor is *flipped* in an engine when its sign differs from the sensor's majority sign.

1. Pairwise flip agreement between "mixed" sensors: share of engines where both are flipped
   or both are not, against the share expected if flips were independent (product of the
   two flip rates). Excess = observed − expected, paired bootstrap over engines.
2. KMeans on the sign vectors (±1; Euclidean distance on ±1 vectors is proportional to
   Hamming distance). k = 1 vs k = 2 by a permutation test: statistic = share of the total
   within-sum-of-squares removed by a 2-way split; null = each sensor's signs permuted
   independently across engines (keeps every sensor's flip rate, destroys co-occurrence).
   If the test does not reject k = 1 at α = 1 − CI_LEVEL, k = 1; otherwise k = argmax
   silhouette over 2..`I_K_MAX`.
3. Per cluster: flip share per sensor, lifetime, and the F health-signal onset (F method,
   `MIN_SEG_PRIMARY`), with between-cluster differences.

In [ ]:
N_PERM = 200  # I: permutations for the k = 1 vs k = 2 test
I_K_MAX = 8  # I: largest k in the silhouette scan


def km_fit(X: np.ndarray, k: int) -> KMeans:
    return KMeans(n_clusters=k, n_init=AUDIT_KMEANS_N_INIT, random_state=AUDIT_KMEANS_SEED).fit(X)


def ordered_labels(X: np.ndarray, k: int) -> np.ndarray:
    """KMeans labels renumbered by cluster size, largest first (C1)."""
    raw = km_fit(X, k).labels_
    order = np.argsort(-np.bincount(raw), kind="stable")
    return np.argsort(order)[raw]


def split_gain(X: np.ndarray) -> float:
    w1 = float(((X - X.mean(axis=0)) ** 2).sum())
    return 0.0 if w1 == 0 else 1.0 - km_fit(X, 2).inertia_ / w1


def boot_diff_median(a: np.ndarray, b: np.ndarray, rng) -> tuple[float, float, float]:
    """Median(a) − median(b), CI resampling engines within each group independently."""
    ia = rng.integers(0, len(a), size=(N_BOOT, len(a)))
    ib = rng.integers(0, len(b), size=(N_BOOT, len(b)))
    bs = np.median(a[ia], axis=1) - np.median(b[ib], axis=1)
    lo, hi = np.quantile(bs, [ALPHA / 2, 1 - ALPHA / 2])
    return float(np.median(a) - np.median(b)), float(lo), float(hi)


def ci3(t, p=2) -> str:
    return f"{t[0]:.{p}f} [{t[1]:.{p}f}, {t[2]:.{p}f}]"


I_sens, I_flip, I_excluded, I_pairs, I_test, I_k, I_sil = {}, {}, {}, [], {}, {}, {}
I_var: dict[str, dict[str, pd.Series]] = {}  # dataset -> {variant name: cluster labels}
for ds in DATASETS:
    rng = rng_for("I", ds)
    rho = E_rho[ds]
    sens = [s for s in E_inf[ds] if rho[s].notna().all()]
    I_excluded[ds] = [s for s in E_inf[ds] if s not in sens]
    sign = np.sign(rho[sens])
    if (sign == 0).any().any():
        raise ValueError(f"{ds}: ρ exactly 0 in some engine — sign undefined")
    flip = pd.DataFrame({s: sign[s] != np.sign(rho[s].median()) for s in sens})
    I_sens[ds], I_flip[ds] = sens, flip

    mixed = [s for s in sens if E_cls[ds, s] == "informative, mixed"]
    n = len(flip)
    for i, a in enumerate(mixed):
        for b in mixed[i + 1 :]:
            fa, fb = flip[a].to_numpy(float), flip[b].to_numpy(float)
            idx = rng.integers(0, n, size=(N_BOOT, n))
            obs_bs = (fa[idx] == fb[idx]).mean(axis=1)
            pa, pb = fa[idx].mean(axis=1), fb[idx].mean(axis=1)
            exc_bs = obs_bs - (pa * pb + (1 - pa) * (1 - pb))
            obs = float((fa == fb).mean())
            exp_ = fa.mean() * fb.mean() + (1 - fa.mean()) * (1 - fb.mean())
            q = [ALPHA / 2, 1 - ALPHA / 2]
            I_pairs.append(
                {
                    "dataset": ds,
                    "pair": f"{a}–{b}",
                    "n engines": n,
                    "engines flipped (first / second)": f"{int(fa.sum())} / {int(fb.sum())}",
                    "agreement [95% CI]": ci3((obs, *np.quantile(obs_bs, q))),
                    "expected if independent": f"{exp_:.2f}",
                    "excess [95% CI]": ci3((obs - exp_, *np.quantile(exc_bs, q))),
                }
            )

    X = sign.to_numpy(float)
    obs_gain = split_gain(X)
    null = np.empty(N_PERM)
    for i in range(N_PERM):
        Xp = np.column_stack([rng.permutation(X[:, j]) for j in range(X.shape[1])])
        null[i] = split_gain(Xp)
    p_perm = (1 + int((null >= obs_gain).sum())) / (N_PERM + 1)
    n_patterns = len(np.unique(X, axis=0))
    sil = {}
    for k in range(2, min(I_K_MAX, n_patterns - 1) + 1):
        sil[k] = float(silhouette_score(X, km_fit(X, k).labels_))
    I_sil[ds] = sil
    k = 1 if p_perm > ALPHA else max(sil, key=sil.get)
    at_limit = bool(sil) and max(sil, key=sil.get) == max(sil)
    I_k[ds] = k
    lab = np.zeros(n, dtype=int) if k == 1 else ordered_labels(X, k)
    I_var[ds] = {} if k == 1 else {f"k={k} (silhouette)": pd.Series(lab, index=flip.index)}
    if k > 2 and at_limit:
        # silhouette still rising at the end of the scan: k is set by the scan limit, so the
        # 2-way split the permutation test validated is reported alongside
        I_var[ds]["k=2 (permutation-tested split)"] = pd.Series(
            ordered_labels(X, 2), index=flip.index
        )
    I_test[ds] = {
        "engines": n,
        "sensors in sign vector": len(sens),
        "mixed sensors": len(mixed),
        "distinct sign patterns": n_patterns,
        "2-way split gain (observed)": f"{obs_gain:.3f}",
        "null gain median [95% range]": (
            f"{np.median(null):.3f} [{np.quantile(null, ALPHA / 2):.3f}, "
            f"{np.quantile(null, 1 - ALPHA / 2):.3f}]"
        ),
        "permutation p": f"{p_perm:.4f}",
        "best silhouette (k)": (
            f"{max(sil.values()):.3f} ({max(sil, key=sil.get)})" if sil else "—"
        ),
        "k scanned": f"2–{max(sil)}" if sil else "—",
        "silhouette argmax at scan limit": "yes" if at_limit else "no",
        "chosen k": k,
        "cluster sizes": " / ".join(str(int(c)) for c in np.bincount(lab)),
    }

I_pairs_df = pd.DataFrame(I_pairs)
I_test_df = pd.DataFrame(I_test).T.reset_index(names="dataset")
display(I_pairs_df)
display(I_test_df)
display(pd.DataFrame({ds: I_sil[ds] for ds in DATASETS}).round(3))

In [ ]:
I_clu_tabs, I_cmp, I_flipnum, I_diff = {}, {}, {}, {}
for ds, var, lab in [(d_, v, lb) for d_ in DATASETS for v, lb in I_var[d_].items()]:
    rng = rng_for("I", ds)
    flip = I_flip[ds]
    clusters = sorted(lab.unique())
    rows = []
    for s in I_sens[ds]:
        row = {"sensor": s, "class": E_cls[ds, s].replace("informative, ", "")}
        for c in clusters:
            v = flip.loc[lab == c, s].to_numpy(float)
            b = boot(v, rng, stat=np.mean)
            I_flipnum.setdefault((ds, var), {}).setdefault(c, {})[s] = b
            row[f"C{c + 1} flip share [95% CI]"] = fmt(b, 2).rsplit(" (", 1)[0]
        rows.append(row)
    I_clu_tabs[ds, var] = pd.DataFrame(rows)
    life = A_life[ds]
    onset = F_fits[ds, MIN_SEG_PRIMARY]
    if (onset["b"] <= 0).any():
        raise ValueError(f"{ds}: F health signal has non-rising engines")
    onset = onset["onset_ttf"]
    rows = []
    for c in clusters:
        u = lab.index[lab == c]
        rows.append(
            {
                "cluster": f"C{c + 1}",
                "engines": len(u),
                "lifetime median [95% CI]": fmt(boot(life.loc[u], rng), 0).rsplit(" (", 1)[0],
                "onset median [95% CI]": fmt(boot(onset.loc[u], rng), 0).rsplit(" (", 1)[0],
            }
        )
    for c in clusters[1:]:
        u0, u1 = lab.index[lab == clusters[0]], lab.index[lab == c]
        dl = boot_diff_median(life.loc[u1].to_numpy(float), life.loc[u0].to_numpy(float), rng)
        do = boot_diff_median(onset.loc[u1].to_numpy(float), onset.loc[u0].to_numpy(float), rng)
        I_diff.setdefault((ds, var), {})[c] = (dl, do)
        rows.append(
            {
                "cluster": f"C{c + 1} − C1",
                "engines": "",
                "lifetime median [95% CI]": ci3(dl, 0),
                "onset median [95% CI]": ci3(do, 0),
            }
        )
    I_cmp[ds, var] = pd.DataFrame(rows)
    display(Markdown(f"**{ds}** — {var}"))
    display(I_clu_tabs[ds, var])
    display(I_cmp[ds, var])


def flip_group(ds: str, var: str, c: int) -> list[str]:
    """Sensors predominantly flipped in cluster c (flip-share CI entirely above 0.5)."""
    return [s for s, b in I_flipnum[ds, var][c].items() if b[1] > 0.5]


def resolved(t: tuple[float, float, float], unit: str) -> str:
    return f"{t[0]:+.0f} {unit} [{t[1]:+.0f}, {t[2]:+.0f}]" + (
        "" if t[1] > 0 or t[2] < 0 else " (CI includes 0)"
    )


I_lines = []
for (ds, var), per_c in I_flipnum.items():
    parts = []
    for c in sorted(per_c):
        g = flip_group(ds, var, c)
        txt = f"C{c + 1} ({int((I_var[ds][var] == c).sum())} engines): flips " + (
            ", ".join(g) or "none"
        )
        if c in I_diff.get((ds, var), {}):
            dl, do = I_diff[ds, var][c]
            txt += f"; vs C1 lifetime {resolved(dl, 'cycles')}, onset {resolved(do, 'cycles')}"
        parts.append(txt)
    I_lines.append(f"- **{ds}, {var}:** " + "; ".join(parts) + ".")
limit_ds = [ds for ds in DATASETS if I_test[ds]["silhouette argmax at scan limit"] == "yes"]
REPORT.append(
    "## I. Engine subpopulations by degradation direction"
    + NL
    + NL
    + "Per engine, the sign of ρ(sensor, ttf) for each informative sensor of E defined in every "
    "engine (left out: "
    + "; ".join(f"{ds} {', '.join(I_excluded[ds])}" for ds in DATASETS if I_excluded[ds])
    + "). *Flipped* = sign opposite to the sensor's majority sign."
    + NL
    + NL
    + "**1. Do the same engines flip across mixed sensors?** Agreement = share of engines whose "
    "flip status matches on both sensors; *expected* = agreement if flips were independent; "
    "excess CI from a paired bootstrap over engines."
    + NL
    + NL
    + md_table(I_pairs_df)
    + NL
    + NL
    + f"**2. Clustering.** KMeans (n_init = {AUDIT_KMEANS_N_INIT}, random_state = "
    f"{AUDIT_KMEANS_SEED}) on the ±1 sign vectors. k = 1 vs k = 2: permutation test "
    f"({N_PERM} permutations, each sensor's signs shuffled independently across engines); "
    f"statistic = share of within-SS removed by a 2-way split. k = argmax silhouette over "
    f"2–{I_K_MAX} (bounded by distinct sign patterns − 1) when k = 1 is rejected."
    + NL
    + NL
    + md_table(I_test_df)
    + NL
    + NL
    + "Silhouette by k:"
    + NL
    + NL
    + md_table(pd.DataFrame({ds: I_sil[ds] for ds in DATASETS}).round(3).reset_index(names="k"))
    + NL
    + NL
    + "The permutation test rejects k = 1 on every dataset (p at its floor, 1/(N_PERM+1)). "
    "Silhouette, however, peaks at the end of the scanned range on "
    + ", ".join(f"{ds} (only k = 2 possible)" if len(I_sil[ds]) == 1 else ds for ds in limit_ds)
    + ": on discrete sign vectors it keeps rising as clusters approach individual sign "
    "patterns, so it does not identify k. Where that leaves k > 2, the 2-way split the "
    "permutation test validated is reported alongside."
    + NL
    + NL
    + "**3. Per cluster.** Flip share per sensor (CI over the cluster's engines), lifetime and "
    "health-signal onset (F method, min_seg = " + str(MIN_SEG_PRIMARY) + "); differences vs "
    "C1 (largest cluster) with CIs resampling within each cluster."
    + NL
    + NL
    + (NL + NL).join(
        f"**{ds} — {var}**"
        + NL
        + NL
        + md_table(I_clu_tabs[ds, var])
        + NL
        + NL
        + md_table(I_cmp[ds, var])
        for (ds, var) in I_clu_tabs
    )
    + NL
    + NL
    + "Summary (a sensor is listed as flipped in a cluster when its flip-share CI lies "
    "entirely above 0.5):"
    + NL
    + NL
    + NL.join(I_lines)
    + NL
    + NL
    + "Only what the data separate is stated: subpopulations defined by which sensors reverse "
    "direction, and whether lifetime/onset differ between them. No physical interpretation is "
    "attached to the clusters." + NL
)

## J. Health-index consequence of the subpopulations

PC1 of the within-regime z of the non-constant KEEP sensors (as in H), fitted on all
engines (pooled) and separately within each cluster from I. PC1 is oriented to rise toward
failure (its loadings on the direction-consistent sensors, signed by their direction, sum
positive). Explained-variance ratio and loadings carry a bootstrap CI over the engines of
that fit (bootstrap eigenvectors sign-aligned to the full-sample one). Per engine, |ρ(PC1,
ttf)| under the cluster's own PC1 vs under the pooled PC1: paired difference over the
cluster's engines.

In [ ]:
def pc1_fit(zs: np.ndarray, units_arr: np.ndarray, members: np.ndarray, orient: np.ndarray, rng):
    S = np.stack([zs[units_arr == u].sum(0) for u in members])
    C2 = np.stack([zs[units_arr == u].T @ zs[units_arr == u] for u in members])
    N = np.array([(units_arr == u).sum() for u in members], float)

    def eig(ix):
        n_, s_, c_ = N[ix].sum(), S[ix].sum(0), C2[ix].sum(0)
        val, vec = np.linalg.eigh(c_ / n_ - np.outer(s_ / n_, s_ / n_))
        return val[-1] / val.sum(), vec[:, -1]

    evr, w = eig(np.arange(len(members)))
    if float(w @ orient) < 0:
        w = -w
    evr_bs, w_bs = np.empty(N_BOOT), np.empty((N_BOOT, len(w)))
    for i in range(N_BOOT):
        e, v = eig(rng.integers(0, len(members), len(members)))
        evr_bs[i], w_bs[i] = e, (v if v @ w >= 0 else -v)
    q = [ALPHA / 2, 1 - ALPHA / 2]
    return (evr, *np.quantile(evr_bs, q)), w, np.quantile(w_bs, q, axis=0)


J_tabs, J_load, J_summary, J_delta = {}, {}, {}, {}
for ds, var, lab in [(d_, v, lb) for d_ in DATASETS for v, lb in I_var[d_].items()]:
    rng = rng_for("J", ds)
    df = train[ds]
    sens = [s for s in KEEP if E_cls[ds, s] != "constant"]
    zs = Z[ds][sens].to_numpy(float)
    units_arr = df["unit"].to_numpy()
    orient = np.array([E_dir.get((ds, s), 0.0) if s in E_cons[ds] else 0.0 for s in sens])
    groups = {"pooled": lab.index.to_numpy()} | {
        f"C{c + 1}": lab.index[lab == c].to_numpy() for c in sorted(lab.unique())
    }
    fits = {g: pc1_fit(zs, units_arr, m, orient, rng) for g, m in groups.items()}

    def rho_abs(w, members, zs=zs, units_arr=units_arr, df=df):
        rows_m = np.isin(units_arr, members)
        pc = pd.DataFrame({"pc1": zs[rows_m] @ w}, index=df.index[rows_m])
        return per_engine_spearman(pc, df["ttf"][rows_m], df["unit"][rows_m])["pc1"].abs()

    rows = []
    for g, m in groups.items():
        own = rho_abs(fits[g][1], m)
        row = {
            "fit": g,
            "engines": len(m),
            "PC1 explained variance [95% CI]": ci3(fits[g][0], 3),
            "median |ρ(own PC1, ttf)| [95% CI]": fmt(boot(own.dropna(), rng), 3).rsplit(" (", 1)[0],
        }
        if g != "pooled":
            pooled = rho_abs(fits["pooled"][1], m)
            row["same engines, pooled PC1"] = fmt(boot(pooled.dropna(), rng), 3).rsplit(" (", 1)[0]
            J_delta[ds, var, g] = boot((own - pooled).dropna(), rng)
            row["paired Δ (own − pooled) [95% CI]"] = fmt(J_delta[ds, var, g], 3).rsplit(" (", 1)[0]
        else:
            row["same engines, pooled PC1"] = "—"
            row["paired Δ (own − pooled) [95% CI]"] = "—"
        rows.append(row)
    J_tabs[ds, var] = pd.DataFrame(rows)
    J_load[ds, var] = pd.DataFrame(
        [
            {
                "sensor": s,
                "class": E_cls[ds, s].replace("informative, ", ""),
                **{
                    g: f"{fits[g][1][j]:+.2f} [{fits[g][2][0][j]:+.2f}, {fits[g][2][1][j]:+.2f}]"
                    for g in groups
                },
            }
            for j, s in enumerate(sens)
        ]
    )
    J_summary[ds, var] = {g: fits[g] for g in groups}
    display(Markdown(f"**{ds}** — {var}"))
    display(J_tabs[ds, var])
    display(J_load[ds, var])

J_lines = []
for (ds, var), fits in J_summary.items():
    sens = [s for s in KEEP if E_cls[ds, s] != "constant"]
    rev, canc = [], []
    for g in [g for g in fits if g != "pooled"]:
        _, wp, qp = fits["pooled"]
        _, wc, qc = fits[g]
        for j, s in enumerate(sens):
            c_sig = qc[0][j] > 0 or qc[1][j] < 0
            p_sig = qp[0][j] > 0 or qp[1][j] < 0
            if c_sig and p_sig and np.sign(wp[j]) != np.sign(wc[j]):
                rev.append(f"{s} in {g} ({wc[j]:+.2f} vs pooled {wp[j]:+.2f})")
            elif c_sig and not p_sig:
                canc.append(f"{s} in {g} ({wc[j]:+.2f} vs pooled {wp[j]:+.2f}, CI incl. 0)")
    tab = J_tabs[ds, var].set_index("fit")
    gains = [
        f"{g} {tab.loc[g, 'paired Δ (own − pooled) [95% CI]']}" for g in tab.index if g != "pooled"
    ]
    J_lines.append(
        f"- **{ds}, {var}:** loading sign reversed vs pooled: {'; '.join(rev) or 'none'}. "
        f"Pooled loading not resolved from 0 while the cluster's is: {'; '.join(canc) or 'none'}. "
        f"Paired Δ|ρ| (own − pooled PC1): {'; '.join(gains)}."
    )
REPORT.append(
    "## J. Health-index consequence of the subpopulations"
    + NL
    + NL
    + "PC1 of the within-regime z of the non-constant KEEP sensors, pooled vs fitted within each "
    "cluster of I; oriented to rise toward failure. Explained variance and loadings carry a "
    "bootstrap CI over the engines of that fit; |ρ(PC1, ttf)| is per engine."
    + NL
    + NL
    + (NL + NL).join(
        f"**{ds} — {var}**"
        + NL
        + NL
        + md_table(J_tabs[ds, var])
        + NL
        + NL
        + "Loadings:"
        + NL
        + NL
        + md_table(J_load[ds, var])
        for (ds, var) in J_tabs
    )
    + NL
    + NL
    + "Summary:"
    + NL
    + NL
    + NL.join(J_lines)
    + NL
)

## K. Is the multi-regime "noise growth" in H a normalization artifact?

Per engine, per regime: that regime's subsequence of cycles (raw values; within a regime
raw and within-regime z differ by an affine map, so the ratio is the same), detrended by
its own hinge fit (F method, `K_MIN_SEG`), then std of the residual in the last third vs
the first third. Subsequences shorter than `K_MIN_LEN` are skipped and counted. Per engine,
the geometric mean of the ratio over regimes and non-constant KEEP sensors; median and CI
over engines. Control: FD001/FD003, each engine split into `K_CONTROL_SPLITS` interleaved
subsequences (cycle mod `K_CONTROL_SPLITS`) so subsequence length and spacing resemble the
multi-regime case — any ratio above 1 there comes from the method (e.g. curvature the hinge
does not remove), not from regimes.

In [ ]:
K_MIN_SEG = 5  # K: min points each side of the hinge within a regime subsequence
K_MIN_LEN = 15  # K: shortest subsequence analysed (3 thirds of ≥ 5 points)
K_CONTROL_SPLITS = 6  # K: interleaved splits for the single-regime control (= regimes in FD002)


def hinge_resid(t: np.ndarray, y: np.ndarray, min_seg: int) -> np.ndarray:
    f = hinge_fit(t, y, min_seg)
    h = np.maximum(0.0, t - f["tau"])
    a = y.mean() - f["b"] * h.mean()
    return y - (a + f["b"] * h)


K_rows, K_eng = [], {}
for ds in DATASETS:
    rng = rng_for("K", ds)
    df = train[ds]
    sens = [s for s in KEEP if E_cls[ds, s] != "constant"]
    if ds in MULTI_REGIME:
        part = pd.Series(REG[ds], index=df.index)
        how = f"{AUDIT_K[ds]} regimes"
    else:
        part = (df["cycle"] % K_CONTROL_SPLITS).rename(None)
        how = f"control: {K_CONTROL_SPLITS} interleaved splits"
    per_eng, k_short, k_zero = [], 0, 0
    for _, g in df.groupby("unit"):
        logs = []
        for _, sub in g.groupby(part.loc[g.index].to_numpy()):
            if len(sub) < K_MIN_LEN:
                k_short += 1
                continue
            t = sub["cycle"].to_numpy(float)
            third = len(sub) // 3
            for s in sens:
                r = hinge_resid(t, sub[s].to_numpy(float), K_MIN_SEG)
                first, last = r[:third].std(ddof=1), r[-third:].std(ddof=1)
                if first == 0 or last == 0:
                    k_zero += 1
                    continue
                logs.append(np.log(last / first))
        if not logs:
            raise ValueError(f"{ds}: an engine has no analysable subsequence")
        per_eng.append(np.exp(np.mean(logs)))
    K_eng[ds] = np.array(per_eng)
    K_rows.append(
        {
            "dataset": ds,
            "partition": how,
            "residual std ratio last/first third [95% CI]": fmt(boot(K_eng[ds], rng), 3),
            "H's diff-based ratio (all cycles, z)": H.loc[
                (H["dataset"] == ds) & (H["family"] == "rolling std"), "value [95% CI] (n engines)"
            ].item(),
            "subsequences skipped (< K_MIN_LEN)": k_short,
            "sensor-subsequences skipped (zero std)": k_zero,
        }
    )
K_df = pd.DataFrame(K_rows)
display(K_df)


# Control pairing from the data: each multi-regime set is compared with the single-regime set
# whose I flip group (2-way split) overlaps it most (Jaccard).
def k2_var(ds: str) -> str:
    return next(v for v in I_var[ds] if v.startswith("k=2"))


def groups_union(ds: str) -> set[str]:
    v = k2_var(ds)
    return {s for c in I_flipnum[ds, v] for s in flip_group(ds, v, c)}


K_ctrl, K_cmp, K_cmp_num = {}, [], {}
for ds in sorted(MULTI_REGIME):
    gm = groups_union(ds)
    jac = {
        c: len(gm & groups_union(c)) / len(gm | groups_union(c))
        for c in DATASETS
        if c not in MULTI_REGIME
    }
    K_ctrl[ds] = max(jac, key=jac.get)
    ctl = K_ctrl[ds]
    dlt = boot_diff_median(K_eng[ds], K_eng[ctl], rng_for("K", ds))
    K_cmp_num[ds] = dlt
    K_cmp.append(
        {
            "dataset": ds,
            "control": ctl,
            "flip-group Jaccard with control": f"{jac[ctl]:.2f}",
            "within-regime ratio − control ratio [95% CI]": ci3(dlt, 3),
        }
    )
K_cmp_df = pd.DataFrame(K_cmp)
display(K_cmp_df)
REPORT.append(
    "## K. Multi-regime noise growth: artifact check"
    + NL
    + NL
    + f"Per engine and regime, the regime's own subsequence of cycles (≥ {K_MIN_LEN} cycles), "
    f"detrended by its own hinge fit (min_seg = {K_MIN_SEG}); ratio of residual std, last third "
    "vs first third; geometric mean over regimes and non-constant KEEP sensors per engine. "
    f"Control: single-regime sets split into {K_CONTROL_SPLITS} interleaved subsequences "
    "(cycle mod " + str(K_CONTROL_SPLITS) + "), so length and spacing are comparable; the "
    "control measures what the method itself yields without regimes."
    + NL
    + NL
    + md_table(K_df)
    + NL
    + NL
    + "Each multi-regime set against the single-regime control whose flip group in I overlaps "
    "it most:"
    + NL
    + NL
    + md_table(K_cmp_df)
    + NL
    + NL
    + " ".join(
        f"{r['dataset']}: within-regime ratio − control = "
        f"{r['within-regime ratio − control ratio [95% CI]']} — "
        + (
            "below the control, so no noise growth beyond what the method shows without regimes."
            if K_cmp_num[r["dataset"]][2] < 0
            else "no excess over the control resolved (CI includes 0)."
            if K_cmp_num[r["dataset"]][1] <= 0
            else "above the control: residual noise growth not explained by the method."
        )
        for r in K_cmp
    )
    + " H's diff-based ratios on all cycles (FD002 / FD004 above) therefore do not measure "
    "noise growth where the within-regime ratio is not above its control; the controls' own "
    "ratios above 1 show the method's bias (curvature the hinge leaves in the late third)." + NL
)

## L. FD002 s8/s13: is the direction regime-dependent?

Per regime, per engine: Spearman ρ between the raw sensor and ttf over that engine's
cycles in that regime (raw ≡ within-regime z inside a regime). Median ρ and the share of
engines with ρ < 0 (sensor rises toward failure, the direction FD001 shows) per regime,
with CIs over engines. FD004 s8/s13 (also mixed) alongside; FD001 as the single-regime
reference. Regimes are labelled by their raw operating-setting centers (D).

In [ ]:
L_SENS = ["s8", "s13"]
L_rows, L_num = [], {}
for ds in ("FD001", "FD002", "FD004"):
    rng = rng_for("L", ds)
    df = train[ds]
    if AUDIT_K[ds] == 1:
        names = {0: "single regime"}
    else:
        c = pd.DataFrame(
            D_scaler[ds].inverse_transform(D_fit[ds][AUDIT_K[ds]].cluster_centers_),
            columns=["op1", "op2", "op3"],
        )
        names = {r: f"op1≈{c.loc[r, 'op1']:.0f}, op3≈{c.loc[r, 'op3']:.0f}" for r in c.index}
    reg = pd.Series(REG[ds], index=df.index)
    for r in sorted(names, key=lambda r: names[r]):
        sub = df[reg == r]
        rho = per_engine_spearman(sub[L_SENS], sub["ttf"], sub["unit"])
        for s in L_SENS:
            v = rho[s].dropna()
            L_num[ds, names[r], s] = (boot(v, rng), boot((v < 0).astype(float), rng, stat=np.mean))
            L_rows.append(
                {
                    "dataset": ds,
                    "regime": names[r],
                    "sensor": s,
                    "median ρ(sensor, ttf) [95% CI]": fmt(boot(v, rng), 3),
                    "share ρ < 0 [95% CI]": fmt(
                        boot((v < 0).astype(float), rng, stat=np.mean), 2
                    ).rsplit(" (", 1)[0],
                    "engines with ρ undefined": int(rho[s].isna().sum()),
                }
            )
L_df = pd.DataFrame(L_rows)
display(L_df)

L_lines = []
for ds in ("FD002", "FD004"):
    for s_ in L_SENS:
        keys = [k for k in L_num if k[0] == ds and k[2] == s_]
        cons = [k[1] for k in keys if L_num[k][1][0] >= SIGN_SHARE]
        mixd = [k[1] for k in keys if L_num[k][1][0] < SIGN_SHARE]
        neg = [k[1] for k in keys if L_num[k][0][2] < 0]
        L_lines.append(
            f"- **{ds} {s_}:** ≥ {SIGN_SHARE:.0%} of engines share the falling-ttf direction "
            f"(ρ < 0) in {'; '.join(cons) or 'no regime'} — below that in "
            f"{'; '.join(mixd) or 'no regime'}. Median ρ CI entirely below 0 in "
            f"{len(neg)}/{len(keys)} regimes."
        )
REPORT.append(
    "## L. Is the direction of s8/s13 regime-dependent?"
    + NL
    + NL
    + "Per regime, per engine: Spearman ρ between the raw sensor and ttf over that engine's "
    "cycles in that regime (raw ≡ within-regime z inside a regime). FD001 is the single-regime "
    "reference; FD004 s8/s13 are also mixed (E)."
    + NL
    + NL
    + md_table(L_df)
    + NL
    + NL
    + NL.join(L_lines)
    + NL
)

## Decision register vs this audit

Every item in `docs/decisions.md` → supported / contradicted / inconclusive, with the section
and number it rests on. "Inconclusive (out of scope)" marks items no data property can
test (model, tooling and serving choices). The asserts guard each verdict's premise, so a
rerun whose numbers no longer support a verdict fails here instead of printing it. Statuses
in `docs/decisions.md` are **not** changed by this notebook.

In [ ]:
def erow(ds: str, s: str) -> pd.Series:
    return E_tab[ds].set_index("sensor").loc[s]


def hval(ds: str, family: str, sub: str = "") -> str:
    m = H[
        (H["dataset"] == ds)
        & (H["family"] == family)
        & H["measurement"].str.contains(sub, regex=False)
    ]
    if len(m) != 1:
        raise KeyError((ds, family, sub))
    return m.iloc[0]["value [95% CI] (n engines)"].rsplit(" (", 1)[0]


def per_ds(fn) -> str:
    return "; ".join(f"{ds} {fn(ds)}" for ds in DATASETS)


MULTI = [ds for ds in DATASETS if ds in MULTI_REGIME]
OOS = "inconclusive (out of scope)"
D02A_LIST = ["s1", "s5", "s6", "s10", "s16", "s18", "s19"]
fd1_const = [s for s in SENSORS if E_cls["FD001", s] == "constant"]

# premises of the verdicts below
assert all(Fq[ds][0][0.75][0] < RUL_CAP for ds in DATASETS)
assert set(fd1_const) == set(D02A_LIST) - {"s6"} and E_cls["FD001", "s6"].startswith("informative")
assert all(E_cls[ds, s].startswith("informative") for ds in DATASETS for s in KEEP)
assert all(E_d03[ds][2][1] > 0 for ds in MULTI)
assert all(D_kstar[ds] == REGIME_N_CLUSTERS for ds in MULTI)
assert all(D05[ds][0] == 1.0 for ds in MULTI)
assert all(v == 0 for ds in MULTI for k, v in D06[ds].items() if "rows" not in k)
assert all(
    float(C_idx.loc[ds, "KS vs all test [95% CI]"].split("[")[1].split(",")[0]) > 0
    for ds in DATASETS
)
assert all(
    float(hval(ds, "health_index (PCA-1)", "best single").split(" [")[1].split(",")[0]) > 0
    for ds in DATASETS
)
assert all(len(test_rul[ds]) == len(A_tlen[ds]) for ds in DATASETS)

dec = [
    (
        "D01",
        "RUL cap 125",
        "inconclusive",
        f"F: median per-engine onset {per_ds(lambda d: f'{Fq[d][0][0.5][0]:.0f}')} (CIs span "
        f"{med_lo:.0f}–{med_hi:.0f}); 125 lies above p75 on every dataset and at/above the "
        "onset of "
        f"{per_ds(lambda d: f'{Fq[d][1][0]:.0%}')} of engines. Supports a range, not a value.",
    ),
    (
        "D02a",
        "Drop s1, s5, s6, s10, s16, s18, s19 (FD001)",
        "partially contradicted",
        f"E FD001: {', '.join(fd1_const)} single-valued — confirmed. s6 is not: informative, "
        f"consistent direction, median |ρ| {erow('FD001', 's6')['median |ρ| [95% CI]']} vs noise "
        f"baseline {erow('FD001', 's6')['noise baseline median |ρ| [95% CI]']}; constant within "
        f"{erow('FD001', 's6')['engines with ρ undefined']} of 100 engines.",
    ),
    (
        "D02b",
        "Same 14-sensor KEEP for FD002–FD004",
        "partially contradicted",
        "E: every KEEP sensor is informative on all four datasets (supported); informative sensors "
        "missing from KEEP: "
        + per_ds(lambda d: ", ".join(E_dis[d]["informative, not in KEEP"]) or "none")
        + ". KEEP sensors whose direction differs between engines: "
        + per_ds(
            lambda d: ", ".join(s for s in KEEP if E_cls[d, s] == "informative, mixed") or "none"
        )
        + ".",
    ),
    (
        "D03",
        "Per-regime z for FD002/FD004, global for FD001/FD003",
        "supported",
        "E: mean |ρ(sensor, ttf)| over KEEP, global → per-regime: "
        + "; ".join(
            f"{ds} {E_d03[ds][0][0]:.3f} → {E_d03[ds][1][0]:.3f} (Δ {fmt(E_d03[ds][2], 3)})"
            for ds in MULTI
        )
        + ". D: FD001/FD003 are single-regime, so global ≡ per-regime. Association only — the "
        "README's claim about model impact is still unmeasured.",
    ),
    (
        "D04a",
        "k = 6 regimes, FD002",
        "supported",
        f"D: silhouette argmax k = {D_kstar['FD002']} ({D_sil['FD002'][6]:.4f}; k=5 "
        f"{D_sil['FD002'][5]:.4f}, k=7 {D_sil['FD002'][7]:.4f}).",
    ),
    (
        "D04b",
        "k = 6 also for FD004",
        "supported",
        f"D: silhouette argmax k = {D_kstar['FD004']} ({D_sil['FD004'][6]:.4f}; next best "
        f"{sorted(D_sil['FD004'].values())[-2]:.4f}); centers match FD002's to 3 decimals.",
    ),
    (
        "D05",
        "KMeans n_init 10, random_state 0, scaled op settings",
        "supported (seed stability)",
        "D: ARI vs random_state 0 across 10 seeds = "
        + "; ".join(f"{ds} {D05[ds][0]:.3f}" for ds in MULTI)
        + ". Raw op SDs differ "
        f"~{raw_spread.loc['FD002'].max() / raw_spread.loc['FD002'].min():.0f}× "
        "(D), consistent with scaling; unscaled clustering not run.",
    ),
    (
        "D06",
        "Normalization fallbacks (std floor, fillna)",
        "inconclusive",
        "D: on the pipeline's train/val split none fires — "
        + "; ".join(
            f"{ds}: unseen val regimes {D06[ds]['val regimes unseen in train']}, zero-std "
            f"cells {D06[ds]['KEEP sensor × regime cells with std = 0']}, min rows/regime "
            f"{D06[ds]['train-split rows per regime (min)']}"
            for ds in MULTI
        )
        + ". Test path not checked (rule 3). Unexercised, so a raise would not break train/val.",
    ),
    (
        "D07",
        "Lean 42-feature set (value, rolling mean, slope)",
        "inconclusive",
        "H: properties outside the set are present — PCA-1 health index beats the best single "
        "sensor's |ρ| by "
        + per_ds(lambda d: hval(d, "health_index (PCA-1)", "best single"))
        + "; a 30-cycle mean gains more |ρ| than a 15-cycle one ("
        + per_ds(
            lambda d: (
                f"{hval(d, 'rolling mean w=30').split(' [')[0]} vs "
                f"{hval(d, 'rolling mean w=15').split(' [')[0]}"
            )
        )
        + "). J: a health index fitted on all engines tracks failure less well in the "
        "minority-direction subpopulations than one fitted within them (largest paired "
        "difference among the permutation-validated 2-way splits: "
        "Δ|ρ| "
        + (lambda k: f"{k[0]} {k[1]} {k[2]}: {fmt(J_delta[k], 3)}")(
            max((k for k in J_delta if k[1].startswith("k=2")), key=lambda k: J_delta[k][0])
        )
        + "). No model comparison, so no verdict on the set itself.",
    ),
    (
        "D08",
        "Rolling window 20",
        "inconclusive",
        "G/H: slope t-ratio (informative KEEP sensors) at w = 20: "
        + per_ds(lambda d: hval(d, f"slope w={WINDOW}"))
        + "; at w = 30: "
        + per_ds(lambda d: hval(d, f"slope w={SEQ_LEN}"))
        + ". Reported, not chosen.",
    ),
    (
        "D09",
        "Exclude min/max/std, FFT, lags, interactions, PCA health index",
        "partially contradicted",
        "H: motivating property present for the PCA health index (PC1 explained variance "
        + per_ds(lambda d: hval(d, "health_index (PCA-1)", "explained"))
        + "). Rolling std: H's multi-regime late/early ratio ("
        + "; ".join(f"{d} {hval(d, 'rolling std')}" for d in sorted(MULTI_REGIME))
        + ") shrinks within regime to "
        + "; ".join(
            f"{r['dataset']} {r['residual std ratio last/first third [95% CI]'].rsplit(' (', 1)[0]}"
            for r in K_rows
            if r["dataset"] in MULTI_REGIME
        )
        + " against single-regime controls ("
        + "; ".join(
            f"{r['dataset']} − {r['control']} {r['within-regime ratio − control ratio [95% CI]']}"
            for r in K_cmp
        )
        + ", K). Absent for min/max (diff skewness "
        + per_ds(lambda d: hval(d, "rolling min/max", "skew"))
        + ") and for first differences "
        "(white noise: lag-1 ac "
        + per_ds(lambda d: hval(d, "first difference", "autocorr"))
        + ").",
    ),
    (
        "D10",
        "LSTM sequence length 30",
        "inconclusive",
        "G/H: slope t-ratio at w = 30: "
        + per_ds(lambda d: hval(d, f"slope w={SEQ_LEN}"))
        + f". A: test engines shorter than {SEQ_LEN}: "
        + per_ds(lambda d: n_short[d])
        + ".",
    ),
    (
        "D11",
        "Left-pad short histories with zeros",
        "inconclusive",
        f"A: padding applies to {per_ds(lambda d: f'{n_short[d]}/{len(A_tlen[d])}')} test engines; "
        "effect on accuracy not measurable without a model.",
    ),
    ("D12", "Split by engine", OOS, "Code property, tested in tests/."),
    (
        "D13",
        "Single fixed 80/20 split",
        "inconclusive",
        "C: validation sets are "
        + per_ds(lambda d: f"{C_idx.loc[d, 'val engines']} engines")
        + "; split variance not measured here.",
    ),
    ("D14", "Normalization fit on train only", OOS, "Code property, tested in tests/."),
    ("D15", "Causal rolling features", OOS, "Code property, tested in tests/."),
    (
        "D16",
        "One prediction per test engine at its last cycle",
        "supported",
        "A: one RUL label per test engine — "
        + per_ds(lambda d: f"{len(test_rul[d])} labels / {len(A_tlen[d])} units")
        + ".",
    ),
    (
        "D17",
        "Validation = make_val_instances (every 5th cycle, RUL < cap)",
        "contradicted",
        "C: KS vs test true RUL "
        + per_ds(lambda d: C_idx.loc[d, "KS vs all test [95% CI]"])
        + "; test engines with RUL ≥ cap (never sampled): "
        + per_ds(lambda d: f"{n_ge[d]}/{len(test_rul[d])}")
        + "; median RUL val vs test "
        + per_ds(lambda d: f"{C_idx.loc[d, 'val median']} vs {C_idx.loc[d, 'test median']}")
        + ".",
    ),
    (
        "D18",
        "Headline = critical-zone RMSE",
        "inconclusive",
        f"B: rests on {per_ds(lambda d: B_crit[d])} test engines ({B_crit_total} of {B_n_total} "
        "pooled) — one residual per engine.",
    ),
    (
        "D19",
        "Buckets 0–25 / 25–50 / 50–100 / 100+",
        "inconclusive",
        "B: test engines per bucket reported; no data property selects the bounds.",
    ),
    ("D20", "Candidate model set", OOS, "Model choice."),
    ("D21", "XGBoost defaults", OOS, "Model choice."),
    ("D22", "LSTM hyperparameters", OOS, "Model choice."),
    (
        "D23",
        "XGBoost tuning protocol",
        "inconclusive",
        "Tuned on make_val_instances, whose RUL distribution differs from test (see D17).",
    ),
    ("D24", "5-seed robustness check", OOS, "Model choice."),
    ("D25", "Ship LSTM", OOS, "Model selection."),
    ("D25a", "comparison runs tagged legacy_test_selection", OOS, "Tooling."),
    ("D25b", "Training pipeline reproducible", OOS, "Tooling."),
    (
        "D26",
        "Confidence band from test-set bucket RMSE",
        "inconclusive",
        f"B: the critical bucket holds {per_ds(lambda d: B_crit[d])} test engines.",
    ),
    ("D27", "Feature state shipped in bundle", OOS, "Serving."),
    ("D28", "Bundle versioning", OOS, "Serving."),
    ("D29", "CPU-only torch in Docker", OOS, "Packaging."),
    ("D30", "cv_results_FD001 (INVALID)", OOS, "Artifact provenance."),
    ("D31", "Asymmetric loss", OOS, "Model choice."),
    ("D32", "DVC local remote", OOS, "Infrastructure."),
    ("D33", "MLflow sqlite backend", OOS, "Infrastructure."),
]
xs = [
    ("X01", "'won untuned against a tuned XGBoost'", OOS, "Model comparison."),
    (
        "X02",
        "Window 20 vs sequence 30 conflated",
        "inconclusive",
        "G reports slope resolution at both lengths; neither is tested by a model.",
    ),
    ("X03", "'never sees test statistics'", OOS, "Pipeline behaviour."),
    (
        "X04",
        "Sensor list derived from FD001 only",
        "confirmed",
        "E: informative-but-excluded sensors differ by dataset (see D02b), and FD001's own list "
        "omits s6.",
    ),
    ("X05", "RUL_CAP duplicated (resolved)", OOS, "Code."),
    (
        "X06",
        "Removed 258-feature family may matter",
        "confirmed (in part)",
        "H: the PCA health-index and longer-window properties exist in the data (see D07, D09).",
    ),
]

## Assemble `reports/data_audit.md` and run the D34 rerun check

Lines marked volatile (timestamps, this rerun check, the D34 row) are excluded when
comparing with a previous run's report; everything else — every number and every figure's
SHA-256 — must match byte for byte, and `repro.mismatches()` must be empty.

In [ ]:
VOL = " <!-- volatile -->"
env_path = REPORT_PATH.with_suffix(".env.json")
prev_md = REPORT_PATH.read_text(encoding="utf-8") if REPORT_PATH.exists() else None
prev_env = json.loads(env_path.read_text(encoding="utf-8")) if env_path.exists() else None
ctx = repro.environment_context(seeds=SEEDS, n_threads=N_THREADS)

header = (
    "# Data audit — C-MAPSS FD001–FD004\n\n"
    "Produced by `notebooks/00_data_audit.ipynb`; every number below is computed from "
    "`data/raw` by that notebook (nothing from prior knowledge of C-MAPSS). Analysis only: "
    "no pipeline code, config or decision status was changed.\n\n"
    "**Test-set access (rule 3):** `test_FD00x.txt` read for truncation lengths only (A); "
    "`RUL_FD00x.txt` for its label distribution only (B, C). No model was fitted and no feature "
    "was computed on test data. All other sections use the training files.\n\n"
    "**Uncertainty (rule 4):** distribution statistics carry a 95% percentile bootstrap CI "
    f"({N_BOOT} resamples of engines, never rows); `n` = engines. Plain counts over the fixed "
    "files are exact and carry no CI.\n\n"
    "**Audit parameters:** "
    + ", ".join(
        f"`{k}` = {v}"
        for k, v in {
            "N_BOOT": N_BOOT,
            "CI_LEVEL": CI_LEVEL,
            "K_RANGE": f"{K_RANGE[0]}–{K_RANGE[-1]}",
            "SIL_SAMPLE": SIL_SAMPLE,
            "VAL_STEP": VAL_STEP,
            "SIGN_SHARE": SIGN_SHARE,
            "MIN_SEG_GRID": MIN_SEG_GRID,
            "MIN_SEG_PRIMARY": MIN_SEG_PRIMARY,
            "TAIL_LEN": TAIL_LEN,
            "SLOPE_WINDOWS": SLOPE_WINDOWS,
            "SMOOTH_WINDOWS": SMOOTH_WINDOWS,
            "EWM_SPAN": EWM_SPAN,
        }.items()
    )
    + ". Seeds are listed under Reproducibility; the hinge fits are seedless (exhaustive grid "
    "+ closed-form least squares).\n"
)
fig_md = "## Figures\n\n" + md_table(
    pd.DataFrame(
        [{"file": f"reports/figures/data_audit/{k}", "sha256": v} for k, v in sorted(FIGS.items())]
    )
)


def assemble(d34_cell: str, rerun_line: str) -> str:
    dec_df = pd.DataFrame(dec, columns=["ID", "decision", "verdict", "evidence (section: number)"])
    dec_md = md_table(dec_df).split("\n")
    dec_md.append(f"| D34 | Report computations reproduce exactly | {d34_cell} |{VOL}")
    x_md = md_table(pd.DataFrame(xs, columns=["ID", "contradiction", "verdict", "evidence"]))
    return "\n\n".join(
        [
            header,
            *REPORT,
            "## Decision register vs this audit\n\nStatuses in `docs/decisions.md` are unchanged; "
            "this table is input for review.\n\n" + "\n".join(dec_md),
            "### Contradictions register\n\n" + x_md,
            fig_md,
            "## Rerun check (D34)\n\n" + rerun_line + VOL,
            repro.render_markdown(ctx),
        ]
    )


def stable(text: str) -> list[str]:
    return [ln for ln in text.splitlines() if VOL not in ln and not ln.startswith("- Generated:")]


draft = assemble("pending", "pending")
if prev_md is None:
    same, env_mm = None, None
    rerun_line = (
        "No previous report at this path — this run is the baseline. Rerun the "
        "notebook in a fresh process to perform the check."
    )
    d34 = "pending | First run; see the rerun check."
else:
    same = stable(prev_md) == stable(draft)
    env_mm = repro.mismatches(prev_env, ctx) if prev_env is not None else ["<no sidecar>"]
    prev_ts = (prev_env or {}).get("timestamp_utc", "?")
    if same and not env_mm:
        rerun_line = (
            f"Reproduced: this run (fresh process) matches the previous run of "
            f"{prev_ts} on every non-volatile line of this report — all numbers and "
            "all figure SHA-256s — and `repro.mismatches()` on the environment "
            "sidecar is empty."
        )
        d34 = "supported | Rerun check below: identical report and environment."
    elif same:
        # Numbers identical, but the environment sidecar differs: report exactly which
        # recorded values differ, so a recording defect is distinguishable from real drift.
        details = []
        for field in env_mm:
            old, new = (prev_env or {}).get(field), ctx.get(field)
            if isinstance(old, dict) and isinstance(new, dict):
                details += [
                    f"`{field}.{k}`: `{old.get(k)}` → `{new.get(k)}`"
                    for k in sorted(set(old) | set(new))
                    if old.get(k) != new.get(k)
                ]
            else:
                details.append(f"`{field}`: `{old}` → `{new}`")
        rerun_line = (
            f"Numbers reproduced: this run (fresh process) matches the previous run of "
            f"{prev_ts} on every non-volatile line of this report — all numbers and all "
            f"figure SHA-256s. Environment sidecar: `repro.mismatches()` flags {env_mm} — "
            + "; ".join(details)
            + "."
        )
        d34 = (
            "supported for the numbers | Rerun check below: identical report and figures; "
            f"the environment sidecar flags {env_mm}."
        )
    else:
        n_diff = sum(a != b for a, b in zip(stable(prev_md), stable(draft), strict=False)) + abs(
            len(stable(prev_md)) - len(stable(draft))
        )
        rerun_line = (
            f"NOT reproduced: {n_diff} non-volatile line(s) differ from the run of "
            f"{prev_ts}; environment mismatches: {env_mm or 'none'}."
        )
        d34 = (
            f"contradicted | Rerun check below: {n_diff} differing line(s), "
            f"env {env_mm or 'equal'}."
        )

final = assemble(d34, rerun_line)
REPORT_PATH.write_text(final, encoding="utf-8")
repro.write_context(ctx, REPORT_PATH)
stack.close()
print(rerun_line)
display(Markdown(final))